# 1. 表題および提出情報

`CELL-ID: SUBMISSION-INFO-01`

**Toward Evaluating Problem Scale in Quantum Computing: Focusing on Operational Constraints in Transportation**
著者：82535190 Takuma Sano
提出物の種別：第三者審査に向けて，再現性と解釈可能性を検証するNotebook
参照したスライド資料：`0712_MDR2_v2_enriched_appendix.pptx`．

# 研究上の前提と開示事項（Research Premises and Disclosure Statement）

`CELL-ID: PREMISES-DISCLOSURE-01`

## 1. 研究の性質

本Notebookは，量子技術を評価する前段階として，輸送アプリケーションで表現すべき運用制約を構造化する**探索的・予備的分析**である．量子アルゴリズムの性能実験，EVRP最適化実験，実物流の検証実験ではない．本Notebookでは，量子回路，量子シミュレータ，量子実機，古典EVRPソルバーを実行していない．実行するのは，公開データから作成し，分析用に固定した入力データからの合成顧客生成，空間クラスタリング，ルートプロキシ構築，制約別の記述的評価，ブートストラップおよびOAT感度分析である．

## 2. 推定対象

- **observational unit:** 生成された合成顧客，または制約評価に用いるroute-condition行．実観測ではない．
- **analytical unit:** 指定された顧客集合，車両数，充電条件，乱数シード（seed）から構成されるルートプロキシ．
- **aggregation unit:** 主結果は評価可能なルート．補助的にscenario-conditionおよびseed単位でも集計する．
- **numerator:** 対象制約について`unmet=True`となった評価可能ルート数．
- **denominator:** 対象制約について必要列が存在し，`evaluated=True`となったルート数．SOCは分母0である．
- **weighting rule:** 主結果はルート加重（route-weighted）であり，各評価可能ルートへ等しい重みを与える．scenario-weightedとseed-weightedは別の推定量として併記する．
- **conditioning assumptions:** 凍結入力，人口加重生成，需要・サービス時間分布，デポ選択，KMeans，最近傍訪問順，道路距離係数，車両仕様，充電条件，一定速度などを固定する．
- **intended interpretation:** 現行の合成モデルとルートプロキシの下で，各運用制約が明示的なモデル表現を必要とする可能性を示す分析上の示唆．
- **prohibited interpretation:** 実際の配送失敗率，最適EVRP解の実行不能率，企業・事業所の運用品質，EV運用可能性全体，または量子計算の優位性として解釈してはならない．

各制約は個別に評価される．したがって，個別制約の未充足率は，全制約を同時に課した場合の共同実行不能率ではない．各率の事象は重複し得るため，複数の未充足率を加算してはならない．

## 3. 合成データの位置づけ

合成顧客は実在する顧客，配送記録，企業，事業所，個人を表さない．標本母集団は，凍結済みe-Stat人口メッシュのうち，人口が正で実装上の本土座標範囲を満たすメッシュである．メッシュ人口 $P_m$ に対し，抽出確率を $p_m=P_m/\sum_jP_j$ とする．各`(customer_count, seed)`内では非復元抽出を用いるため，同一シナリオ内で同じメッシュは重複しない．乱数生成器は`numpy.random.default_rng(seed × 100000 + customer_count)`である．需要は5–30 kg，サービス時間は5–15分の離散一様分布から生成する．

空間表現は選択メッシュの近似重心であり，建物，道路，配送先住所，土地利用を保存しない．人口分布に基づく空間的重みは保存するが，実注文頻度，企業分布，貨物品目，需要相関，時間帯，顧客間依存，配送頻度は保存しない．非復元抽出は，同一メッシュに複数顧客を置かないため，高人口密度メッシュに複数注文が集中する現象を過小表現する可能性がある．

## 4. 対応のあるシナリオ設計

同一の`(customer_count, seed)`で生成された顧客集合を，異なる車両数および充電条件の間で共有する．目的は，条件比較時に顧客配置差を統制することである．このため条件間の観測は独立ではなく，対応のある設計である．同じシード番号は，同一生成規則に用いる識別子であり，実観測単位，実在する配送日，または同一の実世界状況を意味しない．ブートストラップではseedをクラスタ単位として再標本化し，条件間の対応を維持する．

## 5. プロキシの意味

| 要素 | 代理するもの | 代理しないもの |
|---|---|---|
| population mesh | 合成顧客位置の人口ベース空間重み | 注文数，企業数，貨物需要 |
| synthetic customer | 合成された配送地点・需要・サービス時間 | 実在顧客，個人，実注文 |
| depot proxy | 公開物流施設候補の地理的位置 | 実事業者のデポ，能力，所有関係 |
| route proxy | 顧客割当と訪問順を比較する幾何学的構造 | 実道路経路，最短路，最適EVRP解 |
| road-distance multiplier | 直線距離と道路移動距離の差を表す固定倍率 | 実道路網，渋滞，迂回，一方通行 |
| charger candidate | OCM connectionレコードに基づく地理的候補 | 利用可能な充電施設，空き設備 |
| charger condition | 属性・距離閾値による分析用候補選別規則 | 実際の充電サービス水準 |
| baseline vehicle | 1つの公開仕様行に基づく車両シナリオ | 実運用フリート全体，劣化・季節性能 |
| usable driving range | 静的な距離閾値81.2 km | 逐次SOC，重量・気温・道路勾配の影響 |
| operating-time estimate | 距離，一定速度，合成サービス時間の合計 | 渋滞，休憩，待機，時間窓，充電時間を含む実勤務時間 |

デポは顧客集合の重心に最も近い候補から選ぶため，シナリオ間で固定されない可能性がある．したがって顧客条件間の差には，顧客配置だけでなく選択デポの差も含まれ得る．KMeansは緯度経度上の空間クラスタリングであり，積載量，運行時間，航続距離を満たす最適車両割当ではない．最近傍訪問順はgreedyヒューリスティックであり，最短ルートまたは最適EVRP解ではない．

## 6. 充電データの意味

基礎入力の観測単位は**充電施設数ではなくOpen Charge Mapのcharger-connectionレコード**であり，東京都境界へクリップした137 connection行である．候補IDはOCM POI IDとconnection IDを組み合わせて構成する．同一施設が複数connectionを持つ可能性があるため，137件を137施設と解釈してはならない．

本分析はreal-time availability，occupancy，reservation，outage，実車との最終的なconnector compatibility，実効charging power，charging curve，実道路上のdetour route，waiting time，price，営業時間・会員制等のaccess restrictionsを評価しない．条件によって報告connectorラベルと報告powerを選別・簡略計算に使用するが，実際の互換性や供給電力を検証していない．充電候補との地理的近接性は，実際の充電可能性，充電停止の実行可能性，またはSOC実行可能性を保証しない．

## 7. 時間的・地理的整合性

- e-Stat人口メッシュ：2020年国勢調査．ローカル処理スナップショットは2026-07-05時点の研究資産．
- Open Charge Map：取得日2026-07-05．各レコードの更新日は異なり，動的データである．
- P31物流施設候補：対象年度は現在の監査証拠から一意に確認できず`MISSING`．シナリオ用スナップショットは2026-07-11．
- 車両仕様：複数の公開仕様をまとめた2026-07-11スナップショット．各仕様の対象時点・URLは一部`MISSING`．

異なる時点の人口，充電器，物流拠点，車両仕様を組み合わせており，時間的不整合が存在する．「東京都域」はデータごとに同一処理ではない．OCMはN03東京都境界ポリゴンでクリップされたconnection入力を使用する．人口メッシュ生成では，処理済み東京都メッシュに加えて緯度35.45–35.95，経度138.85–140.25の本土範囲フィルタを用いる．P31候補は処理済みmainland Tokyo proxyである．島しょ部はこの本土分析から除外される．分析用緯度経度はWGS84相当のdecimal degreesとして扱うが，各上流原データからのCRS変換過程全体は本Notebookで再実行しない．

## 8. パラメータの状態

| パラメータ | 値 | 分類 | 根拠・状態 |
|---|---:|---|---|
| 平均速度 | 25 km/h | researcher-defined assumption | 観測交通からの較正根拠は`RATIONALE_NOT_VERIFIED` |
| 道路距離係数 | 1.25 | researcher-defined assumption | 実道路比較による根拠は`RATIONALE_NOT_VERIFIED` |
| 使用可能航続距離 | 81.2 km | derived from another parameter | 116 × (0.90−0.20)．逐次SOCではない |
| 積載容量 | 2,000 kg | obtained from public data | 選択車両仕様行．原URL確認は一部`MISSING` |
| 運行時間上限 | 480分 | researcher-defined assumption | 法令・事業データとの対応は`RATIONALE_NOT_VERIFIED` |
| 顧客需要 | 5–30 kg | researcher-defined assumption | 離散一様分布．観測需要ではない |
| サービス時間 | 5–15分 | researcher-defined assumption | 離散一様分布．観測時間ではない |
| 充電条件 | conservative/balanced/broad | researcher-defined assumption | ソースコードに閾値・欠損処理を明示．実サービス区分ではない |
| 感度分析範囲 | low/base/high | researcher-defined assumption | パラメータレジストリ由来．現実的確率は`RATIONALE_NOT_VERIFIED` |
| 地球半径 | 6,371.0088 km | implementation default | Haversine計算用 |
| ブートストラップ回数・seed | 1,000・20260711 | researcher-defined / implementation default | 再現可能性のため固定 |

パラメータ分類は，empirically observed，obtained from public data，obtained from literature，derived from another parameter，researcher-defined assumption，implementation defaultを区別する．本分析の主要パラメータにempirically observedな配送事業データはない．

## 9. 統計的解釈

100個のシードは100件の実運用観測ではなく，同一の合成生成モデルから得た100反復である．ブートストラップ信頼区間が含むのは，固定済み入力，固定済み生成分布，固定済みモデル，固定済みパラメータの下でのシード・クラスタ変動である．データ取得誤差，需要分布の選択，道路距離係数，車両劣化，充電器の可用性，モデル形式，実運用変動は含まない．

ルート加重（route-weighted）集計では車両数が多い条件ほど多くのルートを生成するため，集計値への寄与が大きい．Section 19でルート加重（route-weighted），シナリオ加重（scenario-weighted），シード加重（seed-weighted）を区別して併記する．

## 10. 文献証拠のコーディング

- **search date:** `NOT_DOCUMENTED`
- **search source:** ローカル保存PDF，arXiv URL，DOI URL，既存証拠レジストリ
- **search terms:** `NOT_DOCUMENTED`
- **inclusion criteria:** スライドAppendix E/Fおよび既存レジストリに含まれる量子VRP・ベンチマーキング文献．体系的レビューとしての事前基準は`NOT_DOCUMENTED`．
- **exclusion criteria:** `NOT_DOCUMENTED`
- **coding definitions:** 表現，評価，運用検証を区別し，欠損を0へ変換しない．
- **extraction procedure:** ローカルPDF・抽出テキスト・既存レジストリの照合．ページ・式を確認できない値は参照値として保持する．
- **reviewer:** 文献一次抽出者は`NOT_DOCUMENTED`．本Notebookの監査再構成にはOpenAI Codexを使用．
- **verification procedure:** ローカル一次資料との照合および`SOURCE_NOT_VERIFIED`状態の付与．独立した第二査読者は`NOT_DOCUMENTED`．
- **missing-information policy:** `Not reported`（論文に報告なし），`Not implemented`（実装されていないことを確認），`Not applicable`（適用対象外），`Insufficient information`（判定材料不足）を区別する．

論文に記載がないことだけを，当該制約が実装されていない証拠として扱わない．文献由来の回路幅・qubit数は本Notebookの合成シナリオから導出された値ではなく，4値はいずれも`SOURCE_NOT_VERIFIED`の参照証拠である．

## 11. 検証結果の意味

再生成データと保存済みCSVの一致は，凍結入力と現在のコードの間の**回帰的一貫性**を示す．一致はempirical validity，operational validity，construct validity，optimality，基礎仮定の正しさ，generalizabilityを証明しない．スライド集計値との一致も，同じ計算経路を再現できたことを示すだけである．

## 12. 研究プロセスの開示

| 項目 | 開示内容 |
|---|---|
| analysis author | 82535190 Takuma Sano |
| code author | リポジトリ原コードの著者情報は`NOT_DOCUMENTED`．監査Notebook改訂コードはOpenAI Codex支援を含む |
| data reviewer | `NOT_DOCUMENTED` |
| literature reviewer | `NOT_DOCUMENTED` |
| generative AI / coding assistant | OpenAI CodexをNotebook生成，コード修正，日本語解説，監査支援に使用 |
| human verification procedure | `NOT_DOCUMENTED`．提出前の人手確認が必要 |
| analysis plan / preregistration | `NOT_DOCUMENTED` / `NOT_PREREGISTERED` |
| planned vs post-hoc | 原スライド分析と提出用監査強化を区別．個別変更の事前・事後区分は`NOT_DOCUMENTED` |
| funding | `NOT_DOCUMENTED` |
| conflicts of interest | `NOT_DOCUMENTED` |
| ethics review | `NOT_DOCUMENTED`．合成データと公開集計・施設データを使用するが，正式な審査要否判断は記録されていない |
| personal-data status | 実在個人を表す顧客データは使用しない．公開施設名等を含む可能性はある |
| submission version | `quantum_transport_reproducibility_audit_revised.ipynb` |
| creation date | 2026-07-13．実行日時はRun ManifestにUTCで記録 |
| Git commit | 実行時にSection 7で取得．未コミット変更の有無も記録 |
| authoritative document | 不整合時に優先する文書は`NOT_DOCUMENTED` |

## 前提一覧

| 前提 | 現在の設定 | 証拠 | 解釈への影響 | 状態 |
|---|---|---|---|---|
| 研究種別 | 運用制約を構造化する探索的予備分析 | 研究目的・実行コード | 最適化性能や量子性能を主張できない | `CONFIRMED` |
| 主要推定対象 | 個別制約のルート加重未充足率（route-weighted unmet rate） | 制約評価・集計コード | 実配送失敗率ではなく，率を加算できない | `CONFIRMED` |
| 顧客データ | 人口加重・非復元抽出による合成顧客 | e-Stat処理入力・生成コード | 実顧客・注文へ直接一般化できない | `CONFIRMED` |
| 対応設計 | 顧客集合を車両・充電条件間で共有 | seed・configuration ID | 条件間観測は独立ではない | `CONFIRMED` |
| デポ | 顧客重心に最も近いP31候補 | 実装コード・候補表 | デポ差がシナリオ差へ混入し得る | `CONFIRMED` |
| ルート | KMeans＋最近傍訪問順 | 実装コード・route edges | 最短路・最適EVRP解ではない | `CONFIRMED` |
| 道路距離 | Haversine×1.25 | パラメータレジストリ | 実道路・交通を表現しない | `ASSUMPTION` |
| 充電データ | 137 connectionレコード | OCM入力スキーマ | 137施設や利用可能充電器を意味しない | `CONFIRMED` |
| SOC | 逐次状態遷移なし | 制約レジストリ | EV運用可能性全体を判定できない | `NOT_EVALUATED` |
| データ時点 | 2020人口と2026取得・スナップショット等を結合 | 来歴表 | 時間的不整合を含む | `CONFIRMED` |
| seed | 合成生成モデルの100反復 | 乱数レジストリ | 100件の実観測ではない | `CONFIRMED` |
| ブートストラップ CI | シード・クラスタ変動のみ | ブートストラップ実装 | モデル・運用不確実性を含まない | `CONFIRMED` |
| 文献回路幅 | スライド由来参照値 | 証拠レジストリ | 本シナリオから導出された値ではない | `SOURCE_NOT_VERIFIED` |
| 回帰検証 | 再生成結果と保存CSVの一致 | 自動検証表 | 経験的・運用的妥当性を証明しない | `CONFIRMED` |
| 人手確認 | 手続きの記録なし | 研究プロセス記録 | 提出前に研究者確認が必要 | `NOT_DOCUMENTED` |
| 倫理・資金・COI | 記録なし | 研究プロセス記録 | 提出先要件に応じ追加開示が必要 | `NOT_DOCUMENTED` |

# 研究上の判断と分析行動の記録（Research Reasoning and Action Trail）

`CELL-ID: RESEARCH-REASONING-OVERVIEW-01`

## 記述原則

本節は研究者の内面的思考を創作せず，スライド，コード，データ，出力，Notebook構成および限定的なGit履歴から確認できる行動と判断を再構成する．状態は次の意味で使用する．

- `CONTEMPORANEOUS_RECORD`：当時のスライド，コードコメント，研究資産に明示された判断．
- `DIRECTLY_OBSERVABLE`：現在のコード，データ，出力から直接確認できる行動．
- `RETROSPECTIVE_RECONSTRUCTION`：現在の成果物から再構成した論理であり，当時の思考記録ではない．
- `NOT_DOCUMENTED`：理由または判断を確認できない．
- `UNCERTAIN`：複数の説明が可能で一意に特定できない．

## 研究の問題意識から分析実施までの概要（Research Narrative Overview）

1. **当初の観察（Initial observation）：** スライド2，5–7は，量子技術評価が問題規模，回路幅，解品質等の技術指標だけでは社会実装条件を十分説明しないという問題設定を示す．`CONTEMPORANEOUS_RECORD`
2. **認識した問題（Perceived problem）：** 同程度の問題規模でも，定式化，符号化，補助変数，評価形態により量子資源が異なり，運用要件が表現されているかを別に確認する必要がある．`CONTEMPORANEOUS_RECORD`
3. **研究質問（Research question）：** 輸送固有のインスタンス規模と運用要件をどのように定義すれば，量子資源要件を意味のある形で解釈できるか．`CONTEMPORANEOUS_RECORD`
4. **分析上の要請（Analytical need）：** 問題規模だけでは捉えにくい条件を検討するため，積載量，運行時間，航続距離，充電，SOCを評価対象として扱う方針を採用した．`RETROSPECTIVE_RECONSTRUCTION`
5. **方法の選択（Methodological choice）：** 完全なEVRP最適化ではなく，公開データを空間・技術条件に用い，合成顧客と共通ルートプロキシで個別制約を比較した．行動は`DIRECTLY_OBSERVABLE`，当時の選択理由は一部`RETROSPECTIVE_RECONSTRUCTION`．
6. **実装（Implementation）：** 人口加重顧客，デポプロキシ，KMeans車両割当，最近傍訪問順，Haversine×1.25，条件別充電候補，制約評価を実装した．`DIRECTLY_OBSERVABLE`
7. **整合性の確認（Validation）：** 乱数シード（seed）の固定，件数検査，入力ハッシュ，保存済みCSVとの回帰比較，ブートストラップ，OAT，スライド値照合を実行した．`DIRECTLY_OBSERVABLE`
8. **結果（Result）：** 距離・時間関連制約が現行シナリオで未充足となり，積載は非拘束的，SOCは未評価だった．`DIRECTLY_OBSERVABLE`
9. **解釈（Interpretation）：** 結果は運用要件を定式化へ含めるかを検討するための分析結果であり，配送失敗率やEVRP最適解の実行可能率ではない．`CONTEMPORANEOUS_RECORD`および`RETROSPECTIVE_RECONSTRUCTION`
10. **次の判断（Next decision）：** スライド16は，運用現実性を高める方向と，アプリケーション要件・量子技術段階を接続する枠組みの方向を次段階の判断として残す．`CONTEMPORANEOUS_RECORD`

## 研究動機の連鎖（Research Motivation Chain）

| 段階 | 研究者が観察した事項 | 懸念または不足 | 本研究への反映 | 証拠 | 状態 |
|---:|---|---|---|---|---|
| 1 | 技術進歩が単線的な性能向上として提示され得る | 技術能力，適用条件，社会的判断条件が分離されない | 評価枠組みを技術指標以外へ拡張 | slides 2, 5 | `CONTEMPORANEOUS_RECORD` |
| 2 | 量子ルーティング研究は規模・幅・評価形態を報告 | 運用制約の表現と検証が比較可能でない | scale・representation・evidence gapを区別 | slides 6–9 | `CONTEMPORANEOUS_RECORD` |
| 3 | 回路幅は定式化と符号化で大きく変わる | qubit数だけでは応用上の意味を判定できない | 回路幅を参照証拠として限定的に扱う | slide 7; evidence registry | `CONTEMPORANEOUS_RECORD` |
| 4 | 輸送では顧客，車両，デポ，制約を列挙できる | 社会実装条件が未定義のまま資源推定できない | 輸送を初期適用領域に選ぶ | slides 3–4 | `CONTEMPORANEOUS_RECORD` |
| 5 | EV配送では距離・充電・SOCが問題となる | 通常VRP規模だけではEV固有要件を表せない | EVRP側の探索分析を設定 | slides 8–12 | `RETROSPECTIVE_RECONSTRUCTION` |
| 6 | 実配送注文は研究資産に存在しない | 観測顧客による比較を実行できない | 人口加重合成顧客を生成 | input inventory; source code | `DIRECTLY_OBSERVABLE` |
| 7 | 完全最適化・道路経路・SOC実装がない | 統合実行可能性を評価できない | 共通ルートプロキシと個別制約率に限定 | source code; slide 10 | `DIRECTLY_OBSERVABLE` |
| 8 | 基準結果が仮定へ依存する | 単一設定だけでは頑健性が不明 | seed ブートストラップとOATを実行 | source code; slide 14 | `DIRECTLY_OBSERVABLE`; rationale partly reconstructed |

## 代替方法の位置づけ

以下の「代替方法」は，当時に実際に比較検討した記録がない限り`POTENTIAL_ALTERNATIVE`であり，「研究者が当時棄却した方法」を意味しない．

| 分析上の要請 | 採用した方法 | 代替方法 | 採用しなかった理由 | 解釈への影響 | 状態 |
|---|---|---|---|---|---|
| 顧客配置 | 人口加重合成 | 実配送注文 | 観測注文が研究資産にない．実際の検討記録はない | 外的妥当性が限定 | `POTENTIAL_ALTERNATIVE` |
| 車両割当 | KMeans | 容量・時間・距離制約付き最適化 | 完全EVRP最適化は現行範囲外 | 最適割当ではない | `CONTEMPORANEOUS_RECORD` for scope; alternative `POTENTIAL_ALTERNATIVE` |
| 訪問順 | greedy最近傍 | TSP/VRP最適化 | 採否理由は`RATIONALE_NOT_DOCUMENTED` | 距離はヒューリスティック | `DIRECTLY_OBSERVABLE` |
| 移動距離 | Haversine×1.25 | 道路ネットワーク最短路 | road-network data unavailableと出力に記録 | 実道路距離ではない | `DIRECTLY_OBSERVABLE` |
| デポ | P31最近傍候補 | 実事業者デポ | 実デポデータがない | シナリオ間でデポが変化 | `DIRECTLY_OBSERVABLE` |
| 評価 | 個別制約未充足 | 統合EVRP実行可能性 | SOC・時間窓・充電動態が未実装 | 共同実行不能率を示さない | `DIRECTLY_OBSERVABLE` |
| 不確実性 | seed-cluster ブートストラップ | モデル・パラメータ不確実性の統合 | 現行実装範囲外．採否記録なし | CIの範囲が限定 | `POTENTIAL_ALTERNATIVE` |
| 感度 | OAT | factorial/global sensitivity | 採否理由は`RATIONALE_NOT_DOCUMENTED` | 相互作用を評価しない | `DIRECTLY_OBSERVABLE` |

## 研究過程の全体図（End-to-End Research Logic Diagram）

`CLASSIFICATION: CONCEPTUAL_RECONSTRUCTION`

```text
Observation                  [Slides 2, 5–7 / CLM-S02, CLM-S07]
    ↓
Research concern             [Section 2 / RESEARCH-QUESTION-01]
    ↓
Research question            [Slide 4 / RESEARCH-QUESTION-01]
    ↓
Required evidence            [Sections 8–11 / provenance, parameters, evidence]
    ↓
Methodological choice        [Sections 12–21 / METHOD-* cells]
    ↓
Data construction            [CUSTOMER-GENERATE-01, ROUTE-GENERATE-01]
    ↓
Computation                  [CONSTRAINT-EVALUATE-01, AGGREGATION-CODE-01]
    ↓
Validation                   [RECONCILIATION-CODE-01, VALIDATION-CODE-01]
    ↓
Result                       [Section 26 / constraint_summary.csv]
    ↓
Interpretation               [Sections 27–28]
    ↓
Method revision / next issue [Slide 16 / operational realism vs application-stage framework]
```

## 研究時系列と説明順序の分離

現在のNotebookは第三者が理解しやすい**expository order**へ再構成されている．この順序を，研究者が当初から完全に計画していたresearch chronologyとして扱ってはならない．確認可能な大まかな段階は，Initial framing → Literature and metric review → Problem-scale comparison → Transportation scenario definition → Synthetic data construction → Constraint operationalization → Sensitivity analysis → Reproducibility audit → Current revisionである．正確な日付と各段階の内部順序は，2026-07-13のリポジトリ再構成コミット以外は`NOT_DOCUMENTED`である．


# 2. 研究質問と目的

`CELL-ID: RESEARCH-QUESTION-01`

**研究質問：** 報告された量子資源要件を，輸送問題に固有のインスタンス規模および運用要件と対応づけて解釈するには，両者をどのように定義すべきか．

本計算の目的は，量子資源を解釈する前段階として，合成EVRPシナリオの設定下で未充足となる運用制約を評価対象として整理することである．本研究は，EVRP最適化器の性能，実配送の運用実績，量子優位性を評価しない．

# 2.1 分析枠組みと本Notebookの成果

`CELL-ID: RESEARCH-FRAMEWORK-01`

本研究で扱う中心的な分析単位は量子アルゴリズムそのものではなく，量子資源を推定する前に定義するアプリケーション要件である．分析上は，(i) application requirements（アプリケーション要件），(ii) mathematical representation（数理表現），(iii) quantum formulation and encoding（量子定式化と符号化），(iv) quantum-resource requirements（量子資源要件），(v) feasible technology stage（実行可能な技術段階）を順に区別する．顧客数や車両数だけを問題規模とみなすと，時間，航続距離，充電，SOCを表す変数，制約，補助変数が比較から抜け落ちるため，研究間のqubit数を単純に対応づけることはできない．

本Notebookが提供する主な成果は，公開データと明示した合成仮定を用い，輸送アプリケーションの運用制約と量子資源評価との関係を追跡可能な形で整理した点にある．分析対象は，制約別のルート加重未充足率（route-weighted unmet rate），シナリオ条件による差，仮定変更に対する感度，および量子VRP文献に記載された表現・検証情報との対応である．本研究は探索的な要件整理であり，因果効果，配送事業者母集団への一般化，量子優位性を評価していない．

# 3. 再現範囲

`CELL-ID: SCOPE-01`

**実行モード：`凍結済み処理入力からの計算再現および監査再構成`**

| 再現対象 | 再現方法 | 使用入力 | 状態 | 取得過程を含む再現でない理由 | 第三者が確認できる証拠 |
|---|---|---|---|---|---|
| 合成顧客 | 原実装関数を再実行 | 凍結済みe-Stat処理データ | `REPRODUCED` | e-Stat生データ取得を再実行しない | 行単位比較 |
| ルートと制約 | 原実装関数を再実行 | 凍結済み処理入力 | `REPRODUCED` | 道路ネットワーク最適化ではない | 行単位比較 |
| 集計・ブートストラップ・OAT | 再生成結果から再計算 | 再生成ルート評価 | `REPRODUCED` | モデルとパラメータを固定 | 動的テスト |
| 公開データ取得 | 来歴を監査 | ローカルスナップショットと取得コード | `DERIVED` | 歴史的リクエストが完全保存されていない | ハッシュと処理コード |
| 回路幅参照値 | 参照値として再描画 | スライドとローカル文献 | `REFERENCE_ONLY` | 厳密な導出を確認できない | 証拠レジストリ |
| SOC実行可能性 | 評価なし | 該当なし | `NOT_EVALUATED` | 逐次SOCモデルが存在しない | 制約レジストリ |

# 4. 解釈上の境界

`CELL-ID: BOUNDARY-01`

> 本Notebookで算出する未充足率は，合成シナリオおよびルートプロキシの仮定に条件づけられた分析指標である．実際の配送失敗率，最適化されたEVRP解の実行可能率，または観測された事業運用実績を表すものではない．

ルートプロキシは道路ネットワーク上の経路でも最適化解でもない．航続距離判定はEV運用可能性全体を表さない．`NOT_EVALUATED`を0として扱わない．

# 5. 実行手順

`CELL-ID: EXECUTION-01`

`reproducibility/`ディレクトリでPython 3.11環境を作成し，`requirements-lock.txt`をインストールした後，Notebookを先頭セルから順に実行する．生成物はすべて`outputs/`以下へ保存し，凍結済み入力ファイルは読み取り専用として扱う．

# 6. 事前検査

`CELL-ID: PREFLIGHT-01`

**目的：** 最初の不足項目だけで停止せず，依存関係と入力の不足をすべて収集する
**入力：** リポジトリ判定用ファイル，必須ファイル・列，Pythonモジュール
**処理：** ルート探索，存在・スキーマ・import・Git・書込み検査
**出力：** 事前検査表
**検証：** ERRORレベルの検査がすべて合格すること
**解釈上の境界：** ファイルの可用性は，来歴や科学的妥当性を証明しない

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-PREFLIGHT-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** Notebookの実行前であり，入力と環境の可用性はまだ確認されていない．

**残る疑問（Remaining question）：** 必須ファイル，必要列，Pythonモジュール，Git情報，出力先を利用できるか．

**方法上の判断（Methodological decision）：** 不足項目を個別に停止させず，一覧として検査する．

**分析行動（Action）：** ファイル，列，モジュール，Python，Git，出力先を検査し，事前検査表を作成する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，各検査項目のPASSまたはFAILと，失敗理由を記録した事前検査表．である．

**解釈上の範囲（Interpretation boundary）：** 可用性の検査であり，入力データの科学的妥当性や来歴の完全性は評価しない．

**次の段階（Next step）：** 入力の同一性と来歴を確認する．


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, time, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

START_TIME=datetime.now(timezone.utc)
candidate=Path.cwd().resolve()
support_candidates=[candidate/'src',candidate/'reproducibility/src',candidate.parent/'reproducibility/src']
support_dir=next((p for p in support_candidates if (p/'audit_support.py').is_file()),None)
if support_dir is None:
    raise FileNotFoundError(f'audit_support.py not found from {candidate}; checked {support_candidates}')
sys.path.insert(0,str(support_dir))
from audit_support import *
ROOT=find_repository_root(candidate)
HERE=ROOT/'reproducibility'
OUTPUTS=HERE/'outputs'
FIGURES=OUTPUTS/'figures'; TABLES=OUTPUTS/'tables'; LOGS=OUTPUTS/'logs'; MANIFESTS=OUTPUTS/'manifests'; SYNTH=OUTPUTS/'synthesized'
for directory in [FIGURES,TABLES,LOGS,MANIFESTS,SYNTH]: directory.mkdir(parents=True,exist_ok=True)

required_files={
 'source_deck':(ROOT/'07_presentations/current/0712_MDR2_v2_enriched_appendix.pptx',None),
 'population_mesh':(ROOT/'03_data/processed/estat_tokyo_mesh_population_cells.csv',['mesh_code','total_population']),
 'charger_connections':(ROOT/'03_data/processed/open_charge_map_tokyo_boundary_clipped_connections.csv',['connection_id','latitude','longitude','connection_type','power_kw']),
 'depot_candidates':(ROOT/'03_data/processed/evrp_constraint_gap_inputs/depot_candidates_public_proxy_snapshot.csv',['scenario_depot_id','latitude','longitude','selection_rule']),
 'vehicle_specs':(ROOT/'03_data/processed/evrp_constraint_gap_inputs/vehicle_specs_public_source_snapshot.csv',['scenario_vehicle_id','battery_kwh','catalog_range_km','payload_kg']),
 'analysis_parameters':(ROOT/'03_data/processed/evrp_constraint_gap_inputs/analysis_parameters.csv',['parameter','low','base','high','unit']),
 'quantum_evidence':(ROOT/'03_data/processed/evrp_constraint_gap_inputs/quantum_vrp_evidence_registry.csv',['reference_id','paper_title','url']),
 'stored_customers':(ROOT/'03_data/processed/scenario/synthetic_customers.csv',['customer_configuration_id','seed','customer_id']),
 'stored_routes':(ROOT/'03_data/processed/route_proxy/route_proxy_results.csv',['scenario_id','seed','route_proxy_distance_km']),
 'stored_summary':(ROOT/'03_data/processed/constraints/constraint_summary.csv',['constraint_name','route_weighted_unmet_rate']),
}
required_modules=['numpy','pandas','scipy','sklearn','matplotlib','geopandas','shapely','pyproj','requests','pydantic','pptx','nbformat','pytest']
preflight=preflight_check(ROOT,OUTPUTS,required_files,required_modules)
display(preflight)
preflight.to_csv(TABLES/'preflight_check.csv',index=False)
if preflight.status.eq('FAIL').any():
    raise RuntimeError('Preflight failed. Review all FAIL rows above; no analysis was started.')

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-PREFLIGHT-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** Notebookの実行前であり，入力と環境の可用性はまだ確認されていない．

**残る疑問（Remaining question）：** 必須ファイル，必要列，Pythonモジュール，Git情報，出力先を利用できるか．

**方法上の判断（Methodological decision）：** 不足項目を個別に停止させず，一覧として検査する．

**分析行動（Action）：** ファイル，列，モジュール，Python，Git，出力先を検査し，事前検査表を作成する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，各検査項目のPASSまたはFAILと，失敗理由を記録した事前検査表．である．

**解釈上の範囲（Interpretation boundary）：** 可用性の検査であり，入力データの科学的妥当性や来歴の完全性は評価しない．

**次の段階（Next step）：** 入力の同一性と来歴を確認する．


# 6.1 主要な研究判断の記録（Decision Point Registry）

`CELL-ID: REASONING-REGISTRY-01`

以下の表は，現在のスライド，コード，入力，出力から研究判断を再構成したものである．`RETROSPECTIVE_RECONSTRUCTION`は当時の思考記録ではない．理由を確認できない判断は`RATIONALE_NOT_DOCUMENTED`とする．証拠区分はpublic dataset，literature evidence，code output，visual inspection，regression test，sensitivity result，researcher assumption，implementation constraint，unavailable evidenceを区別する．

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-DECISION-REGISTRY-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 研究上の判断を証拠状態別に記録する方針を定めた．

**残る疑問（Remaining question）：** 主要な方法選択について，確認できる行動と記録のない理由を区別できるか．

**方法上の判断（Methodological decision）：** 18件の判断を共通スキーマで整理し，理由が不明な項目には`RATIONALE_NOT_DOCUMENTED`を付す．

**分析行動（Action）：** Decision Point Registryを作成してCSVへ保存する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，18件の判断，根拠，結果，後続判断を含む`decision_point_registry.csv`．である．

**解釈上の範囲（Interpretation boundary）：** レジストリは現存資料から追跡できる範囲を示すものであり，当時の内面的な思考を復元するものではない．

**次の段階（Next step）：** 研究質問と具体的な方法・分析行動との対応を整理する．


In [ ]:
decision_columns=['decision_id','date_or_stage','question_faced','available_information','options_considered','selected_option','rationale','action_taken','expected_consequence','observed_consequence','validation','subsequent_decision','evidence','status']
decision_rows=[
('D01','Initial framing','なぜ交通・配送を対象としたか','スライド3の適用領域比較','交通; 他領域','交通・配送','問題インスタンスと運用要件を複数水準で定義可能','輸送を初期領域に設定','適用要件と資源解釈を具体化','公開データと制約を構造化','slide trace','EV配送条件へ具体化','slide 3','CONTEMPORANEOUS_RECORD'),
('D02','Scenario definition','なぜEV配送条件を扱うか','range・SOC・charging gap','通常VRP; EVRP','EV配送側の制約','EV固有要件を明示するため（再構成）','航続距離・充電等を評価対象化','運用要件の追加負荷を示す','距離・充電関連の未充足を算出','constraint registry','SOC未評価を分離','slides 8–13; code','RETROSPECTIVE_RECONSTRUCTION'),
('D03','Data construction','なぜ合成顧客か','実注文データなし; 人口メッシュあり','実注文; 合成','人口加重合成','RATIONALE_NOT_DOCUMENTED（可用資産から合理的に再構成可能）','17,500行を生成','比較可能な顧客集合','保存済み結果と一致','regression test','実需要較正を将来課題化','public dataset; code','DIRECTLY_OBSERVABLE'),
('D04','Geographic scope','なぜ東京都域か','東京の人口・充電・P31処理資産','東京; 他地域','東京都本土プロキシ','RATIONALE_NOT_DOCUMENTED','東京入力を統合','共通地理範囲','東京合成シナリオを生成','bounds/hash tests','一般化限界を明記','public dataset','NOT_DOCUMENTED'),
('D05','Factor design','なぜ顧客数25/50/100か','スライド設定','他水準を含む','25/50/100','RATIONALE_NOT_DOCUMENTED','3水準を直積化','規模応答を比較','27構造を生成','count test','規模別集計','slide 18; config','CONTEMPORANEOUS_RECORD'),
('D06','Factor design','なぜ車両数1/3/5か','スライド設定','他水準を含む','1/3/5','RATIONALE_NOT_DOCUMENTED','3水準を直積化','ルート分割応答を比較','1/3/5ルートを生成','route count test','重み付け差を明記','slide 18; config','CONTEMPORANEOUS_RECORD'),
('D07','Randomization','なぜ100 seedか','スライド設定','他反復数','100','RATIONALE_NOT_DOCUMENTED','100顧客集合を生成','合成空間変動を観察','各顧客数100集合','seed completeness test','cluster Bootstrap','slide 18; code','CONTEMPORANEOUS_RECORD'),
('D08','Depot','なぜデポプロキシか','P31候補; 実デポなし','実デポ; 固定点; proxy','最近傍P31候補','RATIONALE_NOT_DOCUMENTED','顧客重心最近傍を選択','各集合に出発点を付与','シナリオでデポが変化','coordinate tests','デポ依存を限界化','public dataset; code','DIRECTLY_OBSERVABLE'),
('D09','Assignment','なぜKMeansか','顧客座標; 車両数','最適割当; random; KMeans','KMeans','RATIONALE_NOT_DOCUMENTED','seed固定n_init=20','空間的に近い顧客を群分け','車両数と同数のクラスタ','route count regression','制約評価へ進む','code','DIRECTLY_OBSERVABLE'),
('D10','Visit order','なぜ最近傍順か','クラスタ内座標','TSP最適化; 道路順; greedy','greedy最近傍','RATIONALE_NOT_DOCUMENTED','デポ往復順を生成','共通距離proxy','非最適距離を生成','stored route comparison','道路・最適性限界を明記','code','DIRECTLY_OBSERVABLE'),
('D11','Distance','なぜ道路ネットワークを使わないか','network distance unavailable記録','network; Haversine','Haversine×係数','network data unavailable; 係数根拠は未確認','距離に1.25を適用','道路迂回を粗く近似','route_proxy_distanceを生成','nonnegative/regression tests','road-networkを次段階へ','code output','DIRECTLY_OBSERVABLE'),
('D12','Constraints','なぜrange/payload/time/accessか','slides 8–13','他制約を含む','個別4領域＋簡略充電','スライド研究範囲','各feasible flagを評価','拘束的要件を識別','制約別率を生成','rate tests','SOCを別扱い','slides; code','CONTEMPORANEOUS_RECORD'),
('D13','Scope limit','なぜSOC/待ちを評価しないか','状態遷移・queueデータなし','実装; 未評価','NOT_EVALUATED','必要実装・データが存在しない','分母0として保持','過剰主張を回避','SOC rateはNaN','T13 test','将来課題へ移す','code; limitation','DIRECTLY_OBSERVABLE'),
('D14','Evaluation unit','なぜルート単位か','車両別route proxy','顧客; scenario; route','route','RATIONALE_NOT_DOCUMENTED','route feasible flagsを作成','車両ルートごとの負荷比較','車両数で寄与が変化','estimand table','他weightingを併記','code','DIRECTLY_OBSERVABLE'),
('D15','Uncertainty','なぜBootstrapか','100 paired seeds','解析式; row bootstrap; cluster bootstrap','seed-cluster Bootstrap','対応設計維持（実装docstring）','1,000回復元抽出','seed変動区間','percentile CI','stored summary comparison','OATへ進む','code comments/output','DIRECTLY_OBSERVABLE'),
('D16','Sensitivity','なぜOATか','low/base/high registry','global/factorial; OAT','OAT','RATIONALE_NOT_DOCUMENTED','一変数ずつ変更','仮定別応答を分離','234行の応答','sensitivity table','相互作用を未解決化','code','DIRECTLY_OBSERVABLE'),
('D17','Literature','なぜ回路幅を残すか','slide 7 values; source uncertainty','削除; 計算値扱い; reference','REFERENCE_ONLY','研究論理との接点を残し過剰主張を避ける（再構成）','証拠レジストリ化','量子側との限定的接続','4値SOURCE_NOT_VERIFIED','status audit','page/equation検証を課題化','slides; local papers','RETROSPECTIVE_RECONSTRUCTION'),
('D18','Audit revision','なぜ再現性監査を追加したか','提出用監査要件; frozen outputs','結果提示のみ; audit','第三者監査Notebook','現ユーザー要求','preflight/manifest/testsを追加','再実行・追跡可能性','18 tests PASS','clean execution','説明と研究時系列を分離','current revision','CONTEMPORANEOUS_RECORD')]
decision_registry=pd.DataFrame(decision_rows,columns=decision_columns)
decision_registry.to_csv(TABLES/'decision_point_registry.csv',index=False)
display(decision_registry)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-DECISION-REGISTRY-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 研究上の判断を証拠状態別に記録する方針を定めた．

**残る疑問（Remaining question）：** 主要な方法選択について，確認できる行動と記録のない理由を区別できるか．

**方法上の判断（Methodological decision）：** 18件の判断を共通スキーマで整理し，理由が不明な項目には`RATIONALE_NOT_DOCUMENTED`を付す．

**分析行動（Action）：** Decision Point Registryを作成してCSVへ保存する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，18件の判断，根拠，結果，後続判断を含む`decision_point_registry.csv`．である．

**解釈上の範囲（Interpretation boundary）：** レジストリは現存資料から追跡できる範囲を示すものであり，当時の内面的な思考を復元するものではない．

**次の段階（Next step）：** 研究質問と具体的な方法・分析行動との対応を整理する．


### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-QUESTION-METHOD-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 主要な研究判断18件を，証拠状態とともに整理した．

**残る疑問（Remaining question）：** 各研究質問が，どの入力，関数，表，解釈へつながるかを追跡できるか．

**方法上の判断（Methodological decision）：** 研究質問ごとに必要な証拠，採用方法，実行内容，出力，残る不確実性を対応づける．

**分析行動（Action）：** Question–Method–Action Mappingを作成する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，研究質問から出力までを対応づけた`question_method_action_mapping.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 対応関係の明示は追跡可能性を高めるが，採用方法の妥当性や代替方法に対する優位性を示さない．

**次の段階（Next step）：** 研究過程の反復と，結果から後続判断への接続を整理する．


In [ ]:
question_method_mapping=pd.DataFrame([
('規模・車両数で制約負荷はどう変わるか','資源解釈前に規模応答が必要','対応付けた顧客集合とroute flags','3×3×3 factorial＋100 seeds','build_analysis_configurations; construct_route_proxies','scenario/estimand tables','モデル条件付き規模応答','水準根拠・外的妥当性'),
('充電条件で地理的アクセスはどう変わるか','EV固有要件の一側面','OCM connection属性と距離','3 screening conditions','build_eligible_charger_candidates; evaluate_routes_by_condition','charger candidates; access rates','地理的候補アクセス','実利用・SOC・queue'),
('どの制約が未充足か','量子定式化へ含める要件を検討','route-level numerator/denominator','個別制約評価','build_constraint_evaluations','constraint_summary','拘束性の分析信号','共同実行可能性'),
('結果はseedで安定するか','合成配置差を把握','paired seed clusters','cluster Bootstrap','cluster_bootstrap_constraint_summary','95% CI','固定モデル下seed変動','モデル不確実性'),
('結果は仮定に依存するか','単一基準値の過剰解釈回避','low/base/high response','OAT','run_oat_sensitivity','sensitivity_detail','局所モデル応答','相互作用')],columns=['research_question','why_it_mattered','required_evidence','selected_method','concrete_action','output','interpretation','remaining_uncertainty'])
question_method_mapping.to_csv(TABLES/'question_method_action_mapping.csv',index=False)
display(question_method_mapping)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-QUESTION-METHOD-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 主要な研究判断18件を，証拠状態とともに整理した．

**残る疑問（Remaining question）：** 各研究質問が，どの入力，関数，表，解釈へつながるかを追跡できるか．

**方法上の判断（Methodological decision）：** 研究質問ごとに必要な証拠，採用方法，実行内容，出力，残る不確実性を対応づける．

**分析行動（Action）：** Question–Method–Action Mappingを作成する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，研究質問から出力までを対応づけた`question_method_action_mapping.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 対応関係の明示は追跡可能性を高めるが，採用方法の妥当性や代替方法に対する優位性を示さない．

**次の段階（Next step）：** 研究過程の反復と，結果から後続判断への接続を整理する．


### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-ITERATION-DECISION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 研究質問と分析行動との対応を表形式で整理した．

**残る疑問（Remaining question）：** 現存資料から確認できる研究範囲の変化と，結果後の判断をどこまで再構成できるか．

**方法上の判断（Methodological decision）：** 日付を推測せず，確認できる段階変化と事後的再構成を分けて記録する．

**分析行動（Action）：** 反復過程表とAnalysis-to-Decision Linksを作成する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`iterative_research_process.csv`と`analysis_to_decision_links.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 記録のない変更理由は研究史として創作せず，`NOT_DOCUMENTED`または事後的再構成として扱う．

**次の段階（Next step）：** 未完了の分析と研究上の仮定を別のレジストリへ記録する．


In [ ]:
iterative_process=pd.DataFrame([
('Framing','技術性能を中心に評価','文献・スライド整理','応用要件が別軸として必要','scale/representation/evidence gapを導入','slides 2–9が変更後構造を示す','slides; exact chronology uncertain'),
('Application definition','問題規模中心','輸送適用条件を整理','大規模でも代表的とは限らない','規模×制約カバレッジへ拡張','slide 9','CONTEMPORANEOUS_RECORD'),
('Exploratory computation','公開データの地理可視化','合成scenario/route/constraint計算','SOC・道路経路・実需要が不足','proxy境界とNOT_EVALUATEDを明示','slides 10–14; code','RETROSPECTIVE_RECONSTRUCTION'),
('Interpretation','主要未充足率を提示','OAT・文献比較','仮定と証拠形態が比較を左右','次段階をoperational realism/frameworkに分岐','slides 14–16','CONTEMPORANEOUS_RECORD'),
('Audit','結果提示中心','再生成・hash・tests','生データ取得と出典に不足','frozen-input reproductionへ限定','current notebook','CONTEMPORANEOUS_RECORD')],columns=['iteration','initial_assumption','analysis_performed','finding_or_problem','revision_made','reason_for_revision','evidence'])
result_links=pd.DataFrame([
('R1','Payload 0%','現設定内で非拘束。設定が緩い可能性もある','モデル内では高; 外部は低','一般に不要とは判断しない','OATと限界記述','需要較正なし'),
('R2','Time 33.5%','時間表現が結果へ影響','モデル内中','時間関連変数を要件候補として維持','感度分析','traffic/break/windowsなし'),
('R3','Range 64.4%','静的距離閾値が頻繁に超過','モデル内中','rangeを省略しない','OAT; SOC課題化','実エネルギーなし'),
('R4','SOC not evaluated','rangeだけでEV実行可能性を判定不能','高','過剰解釈を禁止','NOT_EVALUATED','状態遷移未実装'),
('R5','Circuit values unverified','資源接続は参照段階','低','REFERENCE_ONLYを維持','証拠レジストリ','page/equation不足')],columns=['result_id','result','interpretation','confidence','decision_enabled','action_taken','unresolved_concern'])
iterative_process.to_csv(TABLES/'iterative_research_process.csv',index=False); result_links.to_csv(TABLES/'analysis_to_decision_links.csv',index=False)
display(iterative_process); display(result_links)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-ITERATION-DECISION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 研究質問と分析行動との対応を表形式で整理した．

**残る疑問（Remaining question）：** 現存資料から確認できる研究範囲の変化と，結果後の判断をどこまで再構成できるか．

**方法上の判断（Methodological decision）：** 日付を推測せず，確認できる段階変化と事後的再構成を分けて記録する．

**分析行動（Action）：** 反復過程表とAnalysis-to-Decision Linksを作成する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`iterative_research_process.csv`と`analysis_to_decision_links.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 記録のない変更理由は研究史として創作せず，`NOT_DOCUMENTED`または事後的再構成として扱う．

**次の段階（Next step）：** 未完了の分析と研究上の仮定を別のレジストリへ記録する．


### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-INCOMPLETE-ASSUMPTION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 確認できる反復過程と，結果から後続判断への接続を整理した．

**残る疑問（Remaining question）：** 実行されなかった分析，未評価項目，研究者が置いた仮定を成功結果と分けて提示できるか．

**方法上の判断（Methodological decision）：** 未完了分析と研究者仮定を別表にし，根拠と反証時の影響を記録する．

**分析行動（Action）：** 未完了分析レジストリと仮定ログを作成する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`incomplete_analysis_registry.csv`と`researcher_assumption_log.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 未完了という状態は失敗率0を意味せず，仮定ログは仮定の経験的妥当性を保証しない．

**次の段階（Next step）：** 日付が確認できない段階にはphase名を用いて研究時系列を整理する．


In [ ]:
incomplete_analyses=pd.DataFrame([
('A01','逐次SOC実行可能性','未実装','NOT_EVALUATED','状態・energy/charging eventモデルなし','range結果はSOCを証明しない','constraint registry','INCOMPLETE'),
('A02','道路ネットワーク経路','network mode','未使用','データ unavailable記録','距離はHaversine×係数','route output','INCOMPLETE'),
('A03','古典EVRP最適化baseline','solver comparison','未実行','研究範囲外; solverなし','optimality不明','scope statement','NOT_IMPLEMENTED'),
('A04','観測需要・実配送検証','operational validation','未実行','事業データなし','外的妥当性不明','input inventory','MISSING_DATA'),
('A05','回路幅4値の再導出','paper formula substitution','未完了','page/equation/instance対応未確認','reference only','circuit evidence','SOURCE_NOT_VERIFIED'),
('A06','充電待ち・混雑・価格','operational charger model','未実行','属性・時系列なし','accessは地理のみ','limitations','NOT_IMPLEMENTED'),
('A07','削除・不採用・仮説不一致分析','repository/history review','確認不能','記録なし','研究史を創作しない','Git history limited','NOT_DOCUMENTED')],columns=['attempt_id','intended_purpose','method_attempted','outcome','reason_not_used','implication','evidence','status'])
assumption_log=pd.DataFrame([
('AS1','人口分布を配送需要位置の代理とする','実注文位置がない','population mesh; code','実注文/企業データ','空間結果が変わる',False,'RESEARCHER_ASSUMPTION'),
('AS2','地理距離×1.25で道路距離を近似','network distanceなし','BaselineAssumptions','road shortest path','time/range率が変わる',True,'RATIONALE_NOT_VERIFIED'),
('AS3','空間クラスタを車両割当の代理とする','車両別routeが必要','KMeans code','constrained assignment','route負荷が変わる',False,'RESEARCHER_ASSUMPTION'),
('AS4','最近傍順がroute負荷proxyとなる','共通訪問順が必要','route code','TSP/VRP optimum','distanceが変わる',False,'RESEARCHER_ASSUMPTION'),
('AS5','固定25 km/hで時間を近似','traffic dataを使用しない','config','observed/network time','time率が変わる',True,'RATIONALE_NOT_VERIFIED'),
('AS6','候補距離がcharging accessの一側面','利用データなし','OCM geography','availability model','access解釈が変わる',True,'RESEARCHER_ASSUMPTION'),
('AS7','選択車両仕様をbaselineに使える','比較車両が必要','vehicle snapshot','fleet distribution','range/payloadが変わる',True,'SOURCE_PARTLY_VERIFIED'),
('AS8','個別制約率が実装gapの予備情報になる','統合EVRP未実装','study logic','joint feasibility','要件優先順位が変わる',False,'RETROSPECTIVE_RECONSTRUCTION')],columns=['assumption_id','assumption','why_needed','evidence','alternative','impact_if_false','tested','status'])
incomplete_analyses.to_csv(TABLES/'incomplete_analysis_registry.csv',index=False); assumption_log.to_csv(TABLES/'researcher_assumption_log.csv',index=False)
display(incomplete_analyses); display(assumption_log)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-INCOMPLETE-ASSUMPTION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 確認できる反復過程と，結果から後続判断への接続を整理した．

**残る疑問（Remaining question）：** 実行されなかった分析，未評価項目，研究者が置いた仮定を成功結果と分けて提示できるか．

**方法上の判断（Methodological decision）：** 未完了分析と研究者仮定を別表にし，根拠と反証時の影響を記録する．

**分析行動（Action）：** 未完了分析レジストリと仮定ログを作成する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`incomplete_analysis_registry.csv`と`researcher_assumption_log.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 未完了という状態は失敗率0を意味せず，仮定ログは仮定の経験的妥当性を保証しない．

**次の段階（Next step）：** 日付が確認できない段階にはphase名を用いて研究時系列を整理する．


### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-TIMELINE-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 未完了分析と研究者仮定を，計算結果から分離して記録した．

**残る疑問（Remaining question）：** 研究の進行順と，読者向けに再構成したNotebookの説明順を区別できるか．

**方法上の判断（Methodological decision）：** 確認できる日付だけを使用し，不明な時点は研究段階名で示す．

**分析行動（Action）：** Chronological Research Timelineを作成する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，9段階の`chronological_research_timeline.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 段階の順序は現存資料に基づく再構成を含み，当時の詳細な作業順序を確定するものではない．

**次の段階（Next step）：** 実行環境と凍結入力の来歴を確認する．


In [ ]:
research_timeline=pd.DataFrame([
('Initial framing','技術指標だけで社会実装を評価できるか','slides 2–5','応用要件を先に定義','研究論理を構成','core logic','開始点'),
('Literature and metric review','規模・幅を比較できるか','slides 6–8; papers','gapを3分類','evidence extraction','gap framework','metric focusから拡張'),
('Problem-scale comparison','代表性とは何か','slide 9','規模×制約coverage','application definition','concept figure','規模単独から変更'),
('Transportation scenario definition','要件を計算可能にするには','slides 10–12; public inputs','synthetic proxyを採用','factorial design','27 structures','応用定義を具体化'),
('Synthetic data construction','比較顧客をどう作るか','mesh/code','paired population sampling','17,500 customers','customer CSV','データ作成'),
('Constraint operationalization','どの制約が未充足か','route/code','individual route flags','56,700 evaluations','constraint tables','計算へ移行'),
('Sensitivity analysis','仮定依存はどれか','parameter registry','Bootstrap/OAT','CI/sensitivity','summary tables','単一値から拡張'),
('Reproducibility audit','第三者が再計算できるか','frozen files/code','hash/regression/tests','audit notebook','validation tables','監査層を追加'),
('Current revision','なぜ分析したか追跡できるか','user requirements/artifacts','reasoning/action trail追加','registries/metadata','current notebook','説明責任を追加')],columns=['date_or_phase','question','evidence_available','decision','action','output','change_from_previous_phase'])
research_timeline.to_csv(TABLES/'chronological_research_timeline.csv',index=False); display(research_timeline)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-TIMELINE-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 未完了分析と研究者仮定を，計算結果から分離して記録した．

**残る疑問（Remaining question）：** 研究の進行順と，読者向けに再構成したNotebookの説明順を区別できるか．

**方法上の判断（Methodological decision）：** 確認できる日付だけを使用し，不明な時点は研究段階名で示す．

**分析行動（Action）：** Chronological Research Timelineを作成する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，9段階の`chronological_research_timeline.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 段階の順序は現存資料に基づく再構成を含み，当時の詳細な作業順序を確定するものではない．

**次の段階（Next step）：** 実行環境と凍結入力の来歴を確認する．


# 7. リポジトリおよび実行環境情報

`CELL-ID: ENVIRONMENT-01`

**目的：** 計算環境の同一性を記録する
**入力：** 実行時情報とGit情報
**処理：** Python・パッケージ・OS・Git状態を取得する
**出力：** 環境情報表
**検証：** 未コミット変更を隠さず記録する
**解釈上の境界：** 環境情報の記録だけでは依存関係の固定にならない

In [ ]:
git_info=git_information(ROOT)
packages=['numpy','pandas','scipy','scikit-learn','geopandas','shapely','pyproj','matplotlib','requests','pydantic','python-pptx','nbformat','nbconvert','pytest']
versions=package_versions(packages)
environment=pd.DataFrame([{'python':sys.version,'operating_system':platform.platform(),'architecture':platform.machine(),**git_info}])
display(environment); display(versions)
environment.to_csv(TABLES/'environment.csv',index=False); versions.to_csv(TABLES/'package_versions.csv',index=False)

# 8. 入力データと来歴

`CELL-ID: PROVENANCE-01`

**目的：** 凍結入力を特定し，ファイル同一性を検証する
**入力：** 必須入力ファイルと凍結時の期待SHA-256
**処理：** SHA-256を再計算し期待値と比較する
**出力：** `data_provenance.csv`
**検証：** 完全一致の場合だけ`MATCH`とする
**解釈上の境界：** 処理済みスナップショットから歴史的な取得処理全体は再現できない

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-PROVENANCE-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 事前検査により，実行に必要なファイル，列，モジュール，出力先を検査できる状態にした．

**残る疑問（Remaining question）：** 今回使用する凍結入力が，登録済みのファイルと同一であるか．

**方法上の判断（Methodological decision）：** 各入力のSHA-256を計算し，保存済みハッシュと照合する．

**分析行動（Action）：** 入力ファイルごとにハッシュ，一致状態，取得情報を記録する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`data_provenance.csv`と入力ハッシュのMATCH/MISMATCH状態．である．

**解釈上の範囲（Interpretation boundary）：** ハッシュ一致はファイルの同一性を示すが，データ取得方法，内容の正確性，外的妥当性は示さない．

**次の段階（Next step）：** 照合済み入力からシナリオ構成要素を読み込む．


In [ ]:
expected_hashes={'source_deck': 'c46822f593c4f7b2417bdf220a56e5fe380f30f0aa7a4e25dc02c7b0acb075b1', 'population_mesh': 'cb613e04d74f66d816c4e688f67a59e503e63214ef8410097719429d416e078f', 'charger_connections': '82ea4cea8817f3bd0f00fd6daa5b6964c507d36307dfbebe826f2a466a1a36bb', 'depot_candidates': '839d4bac5e0acf317376e09526e76b8811a6e89c6121c90c53eb829a1601e5b6', 'vehicle_specs': '365e7dd6ba51b17433077afafaf2c34a10ad9c901040f8cdf00007e2a40359a0', 'analysis_parameters': '4d50c6f994bde767611d130c05558a0e3b1cbfe18dffdb889c5720cfcb6f1750', 'quantum_evidence': '1d3bc669a3803beb994b25cd57556bbb0c38588de7fabd2712476e3037f89acb', 'stored_customers': 'aaee0257cf82b6d04e5de812a23220b76267bb5ca742bda0695286d2065d2878', 'stored_routes': '56e737c1dd055681d04eea22297d856d65225860b644f15c0907074111a8fa02', 'stored_summary': 'bde2bba032dc7d7483767c41dbbdf5b2cc40725f83c139941fe5c634bcf98eaa'}
provenance_specs=[
{'dataset_id':'source_deck','formal_name':'Research presentation','provider':'Takuma Sano','source_url':'MISSING','acquisition_date':'2026-07-13','period':'study snapshot','region':'N/A','license':'MISSING','acquisition_method':'local file','api_parameters':'N/A','crs':'N/A','raw_file':'MISSING','preprocessing_script':'N/A','processed_file':str(required_files['source_deck'][0]),'expected_sha256':expected_hashes['source_deck'],'notebook_use':'slides, references, claims','notes':'Corrected source path'},
{'dataset_id':'population_mesh','formal_name':'2020 Population Census Grid Square Statistics','provider':'Statistics Bureau of Japan / e-Stat','source_url':'https://www.e-stat.go.jp/gis/statmap-search?page=1&toukeiCode=00200521&type=1','acquisition_date':'recorded snapshot 2026-07-05','period':'2020 census','region':'Tokyo','license':'MISSING—verify e-Stat terms','acquisition_method':'processed local snapshot','api_parameters':'MISSING','crs':'mesh-code derived WGS84 coordinates in analysis','raw_file':'MISSING/dataless at prior audit','preprocessing_script':'05_src/data_processing/process_tokyo_public_data_inputs.py','processed_file':str(required_files['population_mesh'][0]),'expected_sha256':expected_hashes['population_mesh'],'notebook_use':'population-weighted customer generation','notes':'5,448 input rows'},
{'dataset_id':'charger_connections','formal_name':'Open Charge Map Tokyo clipped connections','provider':'Open Charge Map contributors','source_url':'https://www.openchargemap.org/develop/api','acquisition_date':'2026-07-05','period':'snapshot date','region':'Tokyo N03 boundary','license':'MISSING—verify OCM terms','acquisition_method':'API snapshot then spatial clip','api_parameters':'historical exact request MISSING','crs':'WGS84 latitude/longitude','raw_file':'03_data/raw/charging_infrastructure/open_charge_map/tokyo (availability varies)','preprocessing_script':'05_src/data_processing/fetch_open_charge_map_tokyo.py','processed_file':str(required_files['charger_connections'][0]),'expected_sha256':expected_hashes['charger_connections'],'notebook_use':'charger screening and nearest candidate','notes':'137 connection rows; availability not established'},
{'dataset_id':'depot_candidates','formal_name':'National Land Numerical Information Logistics Facilities P31 proxy','provider':'MLIT Japan','source_url':'https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P31.html','acquisition_date':'snapshot recorded 2026-07-11','period':'MISSING','region':'Tokyo mainland proxy','license':'MISSING—verify MLIT terms','acquisition_method':'processed local snapshot','api_parameters':'N/A','crs':'WGS84 latitude/longitude in snapshot','raw_file':'MISSING/dataless at prior audit','preprocessing_script':'05_src/data_processing/process_mlit_logistics_hubs.py','processed_file':str(required_files['depot_candidates'][0]),'expected_sha256':expected_hashes['depot_candidates'],'notebook_use':'nearest depot proxy','notes':'440 candidates; not operator depots'},
{'dataset_id':'vehicle_specs','formal_name':'Public vehicle specification scenario snapshot','provider':'manufacturer/public pages','source_url':'MISSING—source URLs require verification','acquisition_date':'snapshot 2026-07-11','period':'N/A','region':'Japan','license':'MISSING','acquisition_method':'curated snapshot','api_parameters':'N/A','crs':'N/A','raw_file':'03_data/raw/vehicle_specs/ev_vehicle_specs_sources.csv','preprocessing_script':'05_src/data_processing/process_ev_vehicle_specs.py','processed_file':str(required_files['vehicle_specs'][0]),'expected_sha256':expected_hashes['vehicle_specs'],'notebook_use':'vehicle selection and constraints','notes':'source-chain incomplete'},
]
data_provenance=provenance_table(provenance_specs)
display(data_provenance)
data_provenance.to_csv(TABLES/'data_provenance.csv',index=False)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-PROVENANCE-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 事前検査により，実行に必要なファイル，列，モジュール，出力先を検査できる状態にした．

**残る疑問（Remaining question）：** 今回使用する凍結入力が，登録済みのファイルと同一であるか．

**方法上の判断（Methodological decision）：** 各入力のSHA-256を計算し，保存済みハッシュと照合する．

**分析行動（Action）：** 入力ファイルごとにハッシュ，一致状態，取得情報を記録する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`data_provenance.csv`と入力ハッシュのMATCH/MISMATCH状態．である．

**解釈上の範囲（Interpretation boundary）：** ハッシュ一致はファイルの同一性を示すが，データ取得方法，内容の正確性，外的妥当性は示さない．

**次の段階（Next step）：** 照合済み入力からシナリオ構成要素を読み込む．


# 9. データ辞書

`CELL-ID: DATA-DICTIONARY-01`

**目的：** 主要テーブルの列，型，単位，欠損規則を定義する
**入力：** 主要入力テーブル
**処理：** 実際のスキーマを調査し単位・値域規則を付与する
**出力：** `data_dictionary.csv`
**検証：** 辞書が主要列を網羅することを確認する
**解釈上の境界：** 構造的説明はデータ内容の意味的妥当性を証明しない

In [ ]:
population_mesh=pd.read_csv(required_files['population_mesh'][0])
charger_connections=pd.read_csv(required_files['charger_connections'][0])
depot_candidates=pd.read_csv(required_files['depot_candidates'][0])
vehicle_specs=pd.read_csv(required_files['vehicle_specs'][0])
analysis_parameters=pd.read_csv(required_files['analysis_parameters'][0])
quantum_evidence=pd.read_csv(required_files['quantum_evidence'][0])
input_tables={'input_population_mesh':population_mesh,'input_charger_connections':charger_connections,'input_depot_candidates':depot_candidates,'input_vehicle_specs':vehicle_specs,'input_analysis_parameters':analysis_parameters,'input_quantum_evidence':quantum_evidence}
data_dictionary=build_data_dictionary(input_tables)
data_dictionary.to_csv(TABLES/'data_dictionary.csv',index=False)
display(data_dictionary.head(20))

# 10. 主張・証拠・出力トレーサビリティ表

`CELL-ID: TRACEABILITY-01`

**目的：** スライド，主張，入力，処理，出力を対応付ける
**入力：** 参照スライド資料
**処理：** 22枚のスライドを抽出し安定したセルIDに対応付ける
**出力：** `claim_evidence_traceability.csv`
**検証：** 22枚すべてが含まれること
**解釈上の境界：** 概念的主張は計算結果と区別する

In [ ]:
slides=extract_slide_text(required_files['source_deck'][0])
claim_text={1:'Study identity',5:'Requirements determine representation and resources',7:'Width cannot be interpreted from scale alone',8:'Three gaps coexist',9:'Relevance combines scale and constraint coverage',13:'Constraint results imply modeling requirements',14:'Sensitivity to assumptions',16:'Two future directions'}
traceability=[]
for row in slides.itertuples(index=False):
    claim_id=f'CLM-S{row.slide_number:02d}'
    traceability.append({'claim_id':claim_id,'slide_number':row.slide_number,'slide_element':row.slide_title,'claim_text':claim_text.get(row.slide_number,row.slide_title),'notebook_section':'Sections 1–29','notebook_cell_label':f'TRACE-S{row.slide_number:02d}','input_files':'source deck; relevant frozen inputs','processing_function':'extract_slide_text / study functions','output_table':'claim_evidence_traceability.csv','output_figure':f'figure as applicable to slide {row.slide_number}','evidence_status':'REFERENCE_ONLY' if row.slide_number==7 else 'DERIVED','reproducibility_status':'REPRODUCED' if row.slide_number in [11,12,13,14,17,18,19,20] else 'CONCEPTUAL_SYNTHESIS','limitation':'See section-specific boundary'})
claim_evidence_traceability=pd.DataFrame(traceability)
claim_evidence_traceability.to_csv(TABLES/'claim_evidence_traceability.csv',index=False)
display(claim_evidence_traceability)

# 11. パラメータレジストリ

`CELL-ID: PARAMETERS-01`

**目的：** 結果へ影響する定数を一元管理する
**入力：** ソースコードのdataclass，入力ファイル，スライド設定
**処理：** 出典種別，根拠，不確実性を分類する
**出力：** `parameter_registry.csv`
**検証：** 実行設定との整合性を確認する
**解釈上の境界：** 研究者仮定は観測データで較正された値ではない

# 11.1 パラメータの認識論的位置づけ

`CELL-ID: PARAMETERS-EXPLANATION-01`

パラメータは観測値，公開仕様，研究者仮定，実装既定値，確認済み値からの導出値を区別する．特に道路距離係数1.25，需要5–30 kg，サービス時間5–15分，一定速度25 km/hは，観測配送データから推定されたパラメータではない．これらはモデル内部の比較可能性を確保するための分析条件であり，推定結果の外的妥当性を保証しない．

使用可能航続距離は，選択車両のカタログ航続距離116 kmに対し，初期SOC比率0.90と予備SOC比率0.20の差を適用した導出量である．すなわち，$R_{usable}=116\times(0.90-0.20)=81.2$ kmである．この計算は逐次SOCモデルではなく，全ルート距離と単一閾値を比較するための静的近似である．

In [ ]:
parameter_rows=[
('P01','customer_counts','25; 50; 100','customers','list','scenario','RESEARCHER_ASSUMPTION','slide/config','factorial levels','generation',True,'not calibrated'),('P02','vehicle_counts','1; 3; 5','vehicles','list','scenario','RESEARCHER_ASSUMPTION','slide/config','factorial levels','routing',True,'not calibrated'),('P03','charger_conditions','conservative; balanced; broad','condition','list','scenario','RESEARCHER_ASSUMPTION','source function','screening policies','charger evaluation',True,'attribute missingness'),('P04','n_seeds',100,'seeds','integer','randomness','RESEARCHER_ASSUMPTION','slide/config','spatial variation','all stochastic stages',False,'finite Monte Carlo'),('P05','road_distance_multiplier',1.25,'ratio','float','route','RESEARCHER_ASSUMPTION','BaselineAssumptions','straight-line proxy adjustment','distance/time/range',True,'not calibrated'),('P06','demand_min_max','5–30','kg/customer','integer range','demand','RESEARCHER_ASSUMPTION','BaselineAssumptions','synthetic demand','payload',False,'not observed'),('P07','service_time_min_max','5–15','minutes/customer','integer range','time','RESEARCHER_ASSUMPTION','BaselineAssumptions','synthetic service','operating time',True,'not observed'),('P08','travel_speed',25,'km/h','float','time','RESEARCHER_ASSUMPTION','slide/config','constant proxy speed','operating time',True,'no traffic'),('P09','usable_range',81.2,'km','float','vehicle','DERIVED','116 km × (0.90−0.20)','scenario range','range',True,'source URL incomplete'),('P10','payload_capacity',2000,'kg','float','vehicle','PUBLIC_DATA','vehicle snapshot','selected row','payload',True,'not observed fleet'),('P11','operating_limit',480,'minutes','float','time','RESEARCHER_ASSUMPTION','slide/config','daily proxy limit','operating time',True,'breaks omitted'),('P12','earth_radius',6371.0088,'km','float','geodesy','IMPLEMENTATION_DEFAULT','scenario_utils.py','Haversine mean Earth radius','distance',False,'spherical Earth'),('P13','bootstrap_iterations',1000,'iterations','integer','statistics','RESEARCHER_ASSUMPTION','slide/config','percentile CI','bootstrap',False,'Monte Carlo error'),('P14','bootstrap_seed',20260711,'integer','integer','randomness','IMPLEMENTATION_DEFAULT','analysis code','deterministic bootstrap','bootstrap',False,'none conditional on implementation')]
parameter_registry=pd.DataFrame(parameter_rows,columns=['parameter_id','parameter_name','value','unit','data_type','category','source_type','source','rationale','used_in','sensitivity_tested','uncertainty'])
parameter_registry.to_csv(TABLES/'parameter_registry.csv',index=False)
display(parameter_registry)

# 12. シナリオ設計

`CELL-ID: SCENARIO-DESIGN-01`

**目的：** 要因計画に基づくシナリオを構築する
**入力：** 凍結入力と登録済みパラメータ
**処理：** 充電候補選別，車両選択，要因の直積を作成する
**出力：** 27シナリオ構造
**検証：** IDの一意性と全要因組合せを検査する
**解釈上の境界：** シナリオ設定は分析上の仮定である

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-SOURCE-IMPORTS-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 凍結入力について，登録済みSHA-256との一致を照合した．

**残る疑問（Remaining question）：** 顧客数，車両数，充電条件を比較するための共通シナリオ構造を構成できるか．

**方法上の判断（Methodological decision）：** 確認済みのソース関数を用い，3×3×3×100の要因構成を作る．

**分析行動（Action）：** 充電候補と車両仕様を読み込み，シナリオ設定を列挙する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，充電条件別候補，基準車両，2,700件のシナリオ構成を記録した`scenario_configurations.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 要因構成は研究者が定めた比較設計であり，実際の配送条件の発生頻度を表さない．

**次の段階（Next step）：** 同じ顧客数とシードで共有する合成顧客を生成する．


In [ ]:
for directory in [ROOT/'05_src/scenario_generation',ROOT/'05_src/sensitivity']:
    if str(directory) not in sys.path: sys.path.insert(0,str(directory))
from scenario_utils import BaselineAssumptions, prepare_population_mesh, generate_synthetic_customers, build_charger_condition_definitions, build_eligible_charger_candidates, select_baseline_vehicle, build_analysis_configurations, construct_route_proxies, evaluate_routes_by_condition, EARTH_RADIUS_KM
from monte_carlo_utils import build_constraint_evaluations, build_case_rates, cluster_bootstrap_constraint_summary, run_oat_sensitivity
assumptions=BaselineAssumptions()
CUSTOMER_COUNTS=[25,50,100]; VEHICLE_COUNTS=[1,3,5]; SEEDS=list(range(1,101))
definitions=build_charger_condition_definitions()
charger_candidates,charger_definitions=build_eligible_charger_candidates(charger_connections,definitions)
baseline_vehicle=select_baseline_vehicle(vehicle_specs)
scenario_configurations=build_analysis_configurations(CUSTOMER_COUNTS,VEHICLE_COUNTS,charger_definitions,baseline_vehicle,assumptions,len(SEEDS))
scenario_configurations.to_csv(SYNTH/'scenario_configurations.csv',index=False)
display(scenario_configurations.head())

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-SOURCE-IMPORTS-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 凍結入力について，登録済みSHA-256との一致を照合した．

**残る疑問（Remaining question）：** 顧客数，車両数，充電条件を比較するための共通シナリオ構造を構成できるか．

**方法上の判断（Methodological decision）：** 確認済みのソース関数を用い，3×3×3×100の要因構成を作る．

**分析行動（Action）：** 充電候補と車両仕様を読み込み，シナリオ設定を列挙する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，充電条件別候補，基準車両，2,700件のシナリオ構成を記録した`scenario_configurations.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 要因構成は研究者が定めた比較設計であり，実際の配送条件の発生頻度を表さない．

**次の段階（Next step）：** 同じ顧客数とシードで共有する合成顧客を生成する．


In [ ]:
tracked_functions=[prepare_population_mesh,generate_synthetic_customers,build_eligible_charger_candidates,select_baseline_vehicle,build_analysis_configurations,construct_route_proxies,evaluate_routes_by_condition,build_constraint_evaluations,build_case_rates,cluster_bootstrap_constraint_summary,run_oat_sensitivity]
functions=function_registry(tracked_functions,ROOT,git_info['git_commit'])
functions.to_csv(TABLES/'function_registry.csv',index=False)
display(functions)

# 13. 期待レコード件数の導出

`CELL-ID: COUNT-DERIVATION-01`

シナリオ構造数 ＝ 顧客数3水準 × 車両数3水準 × 充電条件3水準 ＝ **27**．
条件別評価数 ＝ 27構造 × 100個のシード ＝ **2,700**．
合成顧客レコード数 ＝ (25 + 50 + 100)顧客 × 100個のシード ＝ **17,500**．顧客集合を車両数・充電条件間で共有するため，これらの要因を再度乗じない．
条件・ルート評価数 ＝ (1 + 3 + 5ルート) × 顧客数3水準 × 100個のシード × 充電条件3水準 ＝ **8,100**．
ブートストラップ反復数 ＝ 制約ごとに**1,000回**．

In [ ]:
expected_counts=pd.DataFrame([('scenario_structures',3*3*3,27),('conditional_evaluations',27*100,2700),('synthetic_customer_records',(25+50+100)*100,17500),('route_condition_evaluations',(1+3+5)*3*100*3,8100),('bootstrap_iterations',1000,1000)],columns=['quantity','derived_value','expected_value']); expected_counts['status']=np.where(expected_counts.derived_value.eq(expected_counts.expected_value),'PASS','FAIL'); display(expected_counts)

# 14. 合成顧客の生成

`CELL-ID: METHOD-CUSTOMER-01`

**目的：** 対応付けられた合成顧客集合を生成する
**入力：** 人口が正の人口メッシュ
**処理：** 人口加重・非復元抽出，seed規則，需要5–30 kg，サービス時間5–15分
**出力：** 17,500顧客行
**検証：** 確率和，件数，座標範囲，再実行一致，保存済み結果との一致
**解釈上の境界：** 合成位置・需要・サービス時間は観測注文ではない

# 14.1 標本抽出手続きの形式化

`CELL-ID: METHOD-CUSTOMER-FORMAL-01`

人口が正であり，実装上の東京都本土緯度経度範囲を満たすメッシュ集合を $M$ とする．メッシュ $m\in M$ の人口を $P_m$ とすると，抽出確率は次式で定義される．

$$p_m=\frac{P_m}{\sum_{j\in M}P_j},\qquad \sum_{m\in M}p_m=1.$$

各顧客数 $n\in\{25,50,100\}$ とseed $s\in\{1,\ldots,100\}$ に対し，局所乱数生成器 `default_rng(100000s+n)` を生成し，$p_m$ に比例した非復元抽出で $n$ メッシュを選択する．顧客座標は選択メッシュの実装上の近似重心であり，メッシュ内の連続一様点ではない．需要 $d_i$ は離散一様分布 $U\{5,\ldots,30\}$ kg，サービス時間 $q_i$ は $U\{5,\ldots,15\}$ 分から同じ局所生成器で抽出する．

顧客集合は `(customer_count, seed)` ごとに一度だけ生成され，車両数および充電条件の比較で共有される．この対応付けにより条件差を同一顧客集合上で比較できる一方，結果は人口分布，離散需要分布，メッシュ重心近似に条件づけられる．

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-CUSTOMER-GENERATE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 顧客数，車両数，充電条件，シードからなる2,700件のシナリオ構成を定めた．

**残る疑問（Remaining question）：** 実顧客データを用いずに，人口分布を反映した比較用顧客集合を作れるか．

**方法上の判断（Methodological decision）：** 人口メッシュを重みとする非復元抽出を行い，需要とサービス時間を離散一様分布から生成する．

**分析行動（Action）：** 顧客数とシードの組ごとに合成顧客を生成する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，期待件数と主キーを持つ`synthetic_customers.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 顧客位置は人口加重抽出とシードに依存し，実注文の位置，頻度，需要相関を表さない．

**次の段階（Next step）：** 顧客集合に対してデポとルートプロキシを構築する．


In [ ]:
prepared_mesh=prepare_population_mesh(population_mesh,assumptions)
synthetic_customers,randomization_registry=generate_synthetic_customers(population_mesh,CUSTOMER_COUNTS,SEEDS,assumptions)
synthetic_customers.to_csv(SYNTH/'synthetic_customers.csv',index=False)
randomization_registry.to_csv(SYNTH/'randomization_registry.csv',index=False)
stored_customers=pd.read_csv(required_files['stored_customers'][0]); stored_customers['mesh_code']=stored_customers.mesh_code.astype(str)
customers_match,customers_match_detail=compare_frames(synthetic_customers,stored_customers)
display(synthetic_customers.head()); print(customers_match,customers_match_detail)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-CUSTOMER-GENERATE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 顧客数，車両数，充電条件，シードからなる2,700件のシナリオ構成を定めた．

**残る疑問（Remaining question）：** 実顧客データを用いずに，人口分布を反映した比較用顧客集合を作れるか．

**方法上の判断（Methodological decision）：** 人口メッシュを重みとする非復元抽出を行い，需要とサービス時間を離散一様分布から生成する．

**分析行動（Action）：** 顧客数とシードの組ごとに合成顧客を生成する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，期待件数と主キーを持つ`synthetic_customers.csv`．である．

**解釈上の範囲（Interpretation boundary）：** 顧客位置は人口加重抽出とシードに依存し，実注文の位置，頻度，需要相関を表さない．

**次の段階（Next step）：** 顧客集合に対してデポとルートプロキシを構築する．


# 15. デポ選択

`CELL-ID: METHOD-DEPOT-01`

**目的：** 顧客集合ごとにデポプロキシを選ぶ
**入力：** P31由来の440候補
**処理：** 顧客集合重心とのHaversine距離が最小の候補を選択する．同距離の場合は配列上で最初の候補を採用する
**出力：** 顧客集合ごとに1デポ
**検証：** 座標と選択IDを検査する
**解釈上の境界：** 公開物流施設の代理点であり実事業者のデポではない

In [ ]:
display(depot_candidates[['scenario_depot_id','selection_rule','limitation']].head())

# 16. ルートプロキシの構築

`CELL-ID: METHOD-ROUTE-01`

**目的：** 比較可能なルートプロキシを構築する
**入力：** 合成顧客，デポ候補，車両数
**処理：** 乱数シード（seed）の固定KMeans，greedy最近傍訪問，デポ帰着，Haversine距離×1.25，最近傍充電候補探索
**出力：** 基本ルート・辺・構成員・8,100条件別ルート
**検証：** ルート件数，距離非負，保存済み結果との一致
**解釈上の境界：** 最適化解でも道路経路でもなく，クラスタ内訪問順の計算量は概ねO(n²)

# 16.1 ルートプロキシのアルゴリズムと距離定義

`CELL-ID: METHOD-ROUTE-FORMAL-01`

各顧客集合の平均緯度経度を重心とし，その重心までのHaversine距離が最小となるP31候補をデポプロキシとして選択する．車両数 $K>1$ の場合，顧客緯度経度を `KMeans(n_clusters=K, random_state=seed, n_init=20)` で分割する．各クラスタではデポから開始し，未訪問顧客のうち現在地点からHaversine距離が最小の顧客を逐次選択し，最後にデポへ戻る．`numpy.argmin`のため同距離時は配列上で最初の候補が選ばれる．

2点 $(\phi_1,\lambda_1)$，$(\phi_2,\lambda_2)$ 間の距離は，地球半径 $R_E=6371.0088$ kmとして，

$$a=\sin^2\left(\frac{\Delta\phi}{2}\right)+\cos\phi_1\cos\phi_2\sin^2\left(\frac{\Delta\lambda}{2}\right),$$
$$d_H=2R_E\arcsin(\sqrt{a})$$

で計算する．ルート距離プロキシは，往路・顧客間移動・デポ帰着を含むHaversine距離合計に道路距離係数 $\alpha=1.25$ を乗じた $D_r=\alpha\sum_e d_{H,e}$ である．最近傍訪問順の探索はクラスタ当たり概ね $O(n_r^2)$ であり，大規模最適化手法ではない．道路の接続性，一方通行，標高，渋滞，時間窓，SOCを考慮しないため，図示された線は訪問順序の幾何学的表現に限定される．

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-ROUTE-GENERATE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 顧客数とシードの組ごとに，人口加重の合成顧客集合を構成した．

**残る疑問（Remaining question）：** 最適化ソルバーを用いずに，車両数と充電条件の比較に使う共通のルート負荷を計算できるか．

**方法上の判断（Methodological decision）：** デポ候補の選択，KMeansによる割当，最近傍訪問順，Haversine距離への1.25倍補正を用いる．

**分析行動（Action）：** 各シナリオのルート，距離，時間，充電候補への近接性を計算する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`route_results.csv`，ルート構成員，ルート辺の各表．である．

**解釈上の範囲（Interpretation boundary）：** 得られる距離はKMeans，最近傍法，固定距離係数に依存し，道路ネットワークの最短経路や最適EVRP解ではない．

**次の段階（Next step）：** ルート単位で各運用制約を個別に評価する．


In [ ]:
base_routes,route_edges,route_members=construct_route_proxies(synthetic_customers,depot_candidates,CUSTOMER_COUNTS,VEHICLE_COUNTS,SEEDS,assumptions)
route_results=evaluate_routes_by_condition(base_routes,route_members,scenario_configurations,charger_candidates,baseline_vehicle,assumptions)
for name,frame in {'base_routes':base_routes,'route_edges':route_edges,'route_members':route_members,'route_results':route_results}.items(): frame.to_csv(SYNTH/f'{name}.csv',index=False)
stored_routes=pd.read_csv(required_files['stored_routes'][0])
routes_match,routes_match_detail=compare_frames(route_results,stored_routes)
print(route_results.shape,routes_match,routes_match_detail)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-ROUTE-GENERATE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 顧客数とシードの組ごとに，人口加重の合成顧客集合を構成した．

**残る疑問（Remaining question）：** 最適化ソルバーを用いずに，車両数と充電条件の比較に使う共通のルート負荷を計算できるか．

**方法上の判断（Methodological decision）：** デポ候補の選択，KMeansによる割当，最近傍訪問順，Haversine距離への1.25倍補正を用いる．

**分析行動（Action）：** 各シナリオのルート，距離，時間，充電候補への近接性を計算する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`route_results.csv`，ルート構成員，ルート辺の各表．である．

**解釈上の範囲（Interpretation boundary）：** 得られる距離はKMeans，最近傍法，固定距離係数に依存し，道路ネットワークの最短経路や最適EVRP解ではない．

**次の段階（Next step）：** ルート単位で各運用制約を個別に評価する．


In [ ]:
scenario_id='C025_V03_CHG_balanced_SPEED25_T480'; selected_seed=1
sample_edges=route_edges.query('customer_count==25 and vehicle_count==3 and seed==@selected_seed')
sample_results=route_results.query('scenario_id==@scenario_id and seed==@selected_seed')
fig,ax=plt.subplots(figsize=(9,7))
for route_id,edges in sample_edges.groupby('base_route_proxy_id'):
    ordered=edges.sort_values('proxy_edge_order')
    ax.plot(np.r_[ordered.from_longitude.iloc[0],ordered.to_longitude],np.r_[ordered.from_latitude.iloc[0],ordered.to_latitude],marker='o',label=f'{route_id}: {sample_results.loc[sample_results.base_route_proxy_id.eq(route_id),"route_proxy_distance_km"].iloc[0]:.1f} km')
for row in sample_edges[sample_edges.to_node_type.eq('synthetic_customer')].itertuples(): ax.annotate(str(row.proxy_edge_order),(row.to_longitude,row.to_latitude),fontsize=7)
selected_depots=depot_candidates[depot_candidates.scenario_depot_id.isin(sample_results.depot_candidate_id)]
ax.scatter(selected_depots.longitude,selected_depots.latitude,marker='s',s=90,c='black',label='Depot proxy')
ax.scatter(charger_candidates.query('charger_condition=="balanced"').longitude,charger_candidates.query('charger_condition=="balanced"').latitude,marker='^',s=18,facecolors='none',edgecolors='gray',label='Eligible chargers')
chosen=charger_candidates[charger_candidates.charger_candidate_id.isin(sample_results.nearest_candidate_charger_id.dropna())]
ax.scatter(chosen.longitude,chosen.latitude,marker='*',s=130,c='black',label='Selected nearest chargers')
unmet=[]
for r in sample_results.itertuples(): unmet.append(f'R{r.route_index}: '+', '.join(x for x,v in [('time',r.operating_time_feasible),('range',r.range_feasible),('access',r.charger_geographically_accessible)] if v is False))
ax.set(title=f'{scenario_id}; seed={selected_seed}; '+ ' | '.join(unmet),xlabel='Longitude',ylabel='Latitude'); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(FIGURES/'representative_route_proxy.png',dpi=320); fig.savefig(FIGURES/'representative_route_proxy.svg'); plt.show()

# 17. 制約の定義

`CELL-ID: METHOD-CONSTRAINT-01`

**目的：** 各制約の評価意味を定義する
**入力：** ルート結果の各列
**処理：** feasible・evaluated列を分子・分母へ対応付ける
**出力：** `constraint_definitions.csv`
**検証：** `NOT_EVALUATED`を欠損のまま保持する
**解釈上の境界：** 地理的アクセスや静的航続距離はEV運用可能性全体を証明しない

# 17.1 制約の数理的定義

`CELL-ID: METHOD-CONSTRAINT-FORMAL-01`

ルート $r$ の顧客集合を $N_r$，需要を $d_i$，サービス時間を $q_i$，距離を $D_r$ とする．積載未充足指標は $U_{payload,r}=\mathbf{1}[\sum_{i\in N_r}d_i>C]$ である．運行時間は $T_r=(D_r/v)\times60+\sum_{i\in N_r}q_i$ とし，$U_{time,r}=\mathbf{1}[T_r>T_{max}]$ とする．ここで $v=25$ km/h，$T_{max}=480$ 分であり，休憩，待機，充電イベントは含まれない．

航続距離指標は $U_{range,r}=\mathbf{1}[D_r>R_{usable}]$ である．充電アクセスは，ルートプロキシのいずれかのノードから条件別候補までの最近傍地理距離と閾値を比較する．充電支援航続距離と充電時間は，航続距離超過ルートの一部に対する簡略化判定であり，充電地点到着時SOC，充電曲線，待ち時間，営業時間を表現しない．逐次SOC $SOC_{k+1}=SOC_k-E(d_{k,k+1},w_k)+Q_k$ は概念上必要だが，本分析では $E$ と $Q_k$ が実装されていないため，SOCは全件 `NOT_EVALUATED` とする．

In [ ]:
constraint_rows=[
('C-PAYLOAD','Payload capacity','route proxy','unmet evaluated routes','all complete routes','route_total_demand_kg > payload_capacity_kg','route_total_demand_kg; payload_capacity_kg','kg','missing demand/capacity','synthetic demand','capacity signal','real loading rules'),
('C-TIME','Operating-time limit','route proxy','routes above limit','complete distance/speed/service routes','estimated_operating_time_min > operating_time_limit_min','route_proxy_distance_km; assumed_speed_kmh; route_service_time_min','minutes','missing inputs','constant speed; no charging/wait','time signal','traffic, breaks, windows'),
('C-RANGE','Range feasibility','route proxy','routes above usable range','complete route/range','route_proxy_distance_km > usable_range_km','route_proxy_distance_km; usable_range_km','km','missing inputs','fixed usable range','range signal','SOC transition, load/weather'),
('C-SOC','SOC feasibility','route sequence','NOT_EVALUATED','NOT_EVALUATED','NOT_EVALUATED','SOC state trajectory','kWh/SOC','all cases','model absent','none','all sequential SOC behavior'),
('C-ACCESS','Charging-station access','route proxy','no candidate within threshold','all condition routes','nearest_charger_distance_km > threshold','nearest_charger_distance_km; maximum_charger_access_distance_km','km','none','route-node geography','geographic access signal','availability, queue, hours'),
('C-ASSIST','Charging-assisted range','range-infeasible route','unsupported evaluable routes','range-infeasible routes','simplified two-range/access rule fails','distance; range; compatible candidate','km','range-feasible excluded','one simplified support event','support proxy','SOC/stop feasibility'),
('C-DURATION','Charging duration','evaluable assisted route','duration above limit','known positive power and accessible candidate','supplemental duration > limit','energy proxy; power; duration limit','minutes','unknown power/incompatible','constant power','duration proxy','taper, efficiency, queues')]
constraint_registry=pd.DataFrame(constraint_rows,columns=['constraint_id','constraint_name','evaluation_unit','numerator','denominator','unmet_condition','required_columns','unit','excluded_cases','assumptions','interpretation','not_captured'])
constraint_registry.to_csv(TABLES/'constraint_definitions.csv',index=False); display(constraint_registry)

# 18. 制約評価

`CELL-ID: METHOD-EVALUATION-01`

**目的：** 定義済み制約をルート単位で評価する
**入力：** 再生成したルート結果
**処理：** 横持ち結果をevaluated・feasible・unmetの縦持ち形式へ変換する
**出力：** 56,700制約評価行
**検証：** 真偽値の値域と欠損規則を検査する
**解釈上の境界：** 未充足判定は現在のモデル仮定に条件づけられる

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-CONSTRAINT-EVALUATE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 各シナリオについて，ルート距離，積載量，所要時間，充電候補への近接性を計算した．

**残る疑問（Remaining question）：** 各ルートが，定義した積載量，運行時間，航続距離，充電条件を満たすか．

**方法上の判断（Methodological decision）：** 制約ごとに`evaluated`，`feasible`，`unmet`を分け，SOCは未評価のまま保持する．

**分析行動（Action）：** ルート結果を制約別の長形式評価表へ変換する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`constraint_evaluations_long.csv`と，各制約の評価可能件数・未充足件数．である．

**解釈上の範囲（Interpretation boundary）：** 未充足は選択した閾値とルートプロキシに基づく判定であり，実際の配送失敗や全制約を同時に課した実行不能を意味しない．

**次の段階（Next step）：** ルート加重，シナリオ加重，シード加重で集計する．


In [ ]:
constraint_evaluations=build_constraint_evaluations(route_results); case_rates=build_case_rates(constraint_evaluations); constraint_evaluations.to_csv(SYNTH/'constraint_evaluations_long.csv',index=False); case_rates.to_csv(SYNTH/'constraint_case_rates.csv',index=False); display(constraint_evaluations.head())

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-CONSTRAINT-EVALUATE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 各シナリオについて，ルート距離，積載量，所要時間，充電候補への近接性を計算した．

**残る疑問（Remaining question）：** 各ルートが，定義した積載量，運行時間，航続距離，充電条件を満たすか．

**方法上の判断（Methodological decision）：** 制約ごとに`evaluated`，`feasible`，`unmet`を分け，SOCは未評価のまま保持する．

**分析行動（Action）：** ルート結果を制約別の長形式評価表へ変換する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`constraint_evaluations_long.csv`と，各制約の評価可能件数・未充足件数．である．

**解釈上の範囲（Interpretation boundary）：** 未充足は選択した閾値とルートプロキシに基づく判定であり，実際の配送失敗や全制約を同時に課した実行不能を意味しない．

**次の段階（Next step）：** ルート加重，シナリオ加重，シード加重で集計する．


# 19. 集計方法と統計的推定対象

`CELL-ID: METHOD-AGGREGATION-01`

**目的：** route・scenario・seedの重み付けを分離する
**入力：** 縦持ち制約評価
**処理：** 分子，分母，除外数，重み付け別の率を計算する
**出力：** `estimand_comparison.csv`
**検証：** 分子が分母以下であり率が0–1に入ること
**解釈上の境界：** ルート加重（route-weighted）集計では複数ルート条件の寄与が大きい

# 19.1 推定対象の定義

`CELL-ID: METHOD-ESTIMAND-FORMAL-01`

制約 $c$ が評価可能なルート集合を $\mathcal{R}_c$ とし，未充足指標を $U_{r,c}\in\{0,1\}$ とする．主結果のルート加重未充足率（route-weighted unmet rate）は，

$$\hat p_c^{route}=\frac{\sum_{r\in\mathcal{R}_c}U_{r,c}}{|\mathcal{R}_c|}$$

である．この推定量では5車両条件が1車両条件より多くのルートを持つため，より大きな重みを持つ．scenario-weighted率は各 `(scenario_id, seed)` 内のルート率を同じ重みで平均し，seed-weighted率は各シードに属する全評価可能ルートの率を先に計算してシード間で平均する．したがって三者は同じ母数の別表現ではなく，異なる重み付け規則に基づく記述的推定量である．解釈対象は，固定された合成データ生成機構，ルートプロキシ，パラメータ集合の下で生成される分析対象ルートであり，東京都の実配送母集団ではない．

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-AGGREGATION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 個々のルートについて，制約別の評価可否と未充足状態を記録した．

**残る疑問（Remaining question）：** 集計単位の違いが未充足率へどのように反映されるか．

**方法上の判断（Methodological decision）：** ルート，シナリオ，シードをそれぞれ等重みとする3種類の推定対象を分ける．

**分析行動（Action）：** 分子，分母，未充足率を集計単位別に算出する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`estimand_comparison.csv`と集計単位別の率．である．

**解釈上の範囲（Interpretation boundary）：** ルート加重では車両数が多い条件ほど寄与が大きい．異なる重み付けの率は同じ推定対象ではない．

**次の段階（Next step）：** 対応関係を保ったままシード間変動の区間を計算する．


In [ ]:
evaluated=constraint_evaluations[constraint_evaluations.evaluated.astype(bool)].copy(); evaluated['unmet_int']=evaluated.unmet.astype(bool).astype(int)
rows=[]
for constraint,group in constraint_evaluations.groupby('constraint_name'):
    eg=group[group.evaluated.astype(bool)].copy(); route_rate=eg.unmet.astype(bool).mean() if len(eg) else np.nan
    cases=case_rates.query('constraint_name==@constraint').dropna(subset=['case_unmet_rate'])
    seed_counts=eg.assign(unmet_int=eg.unmet.astype(bool).astype(int)).groupby('seed').agg(unmet_count=('unmet_int','sum'),evaluated_count=('unmet_int','size')); seed_rates=seed_counts.unmet_count/seed_counts.evaluated_count
    for method,rate,n in [('route-weighted',route_rate,len(eg)),('scenario-weighted',cases.case_unmet_rate.mean() if len(cases) else np.nan,len(cases)),('seed-weighted',seed_rates.mean() if len(seed_rates) else np.nan,len(seed_rates))]: rows.append({'constraint_name':constraint,'weighting_method':method,'evaluated_count':n,'excluded_count':len(group)-len(eg) if method=='route-weighted' else np.nan,'unmet_count':int(eg.unmet.astype(bool).sum()) if len(eg) else 0,'unmet_rate':rate,'confidence_interval':'computed for route-weighted seed-cluster estimand in Section 20'})
estimand_comparison=pd.DataFrame(rows); estimand_comparison.to_csv(TABLES/'estimand_comparison.csv',index=False); display(estimand_comparison)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-AGGREGATION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 個々のルートについて，制約別の評価可否と未充足状態を記録した．

**残る疑問（Remaining question）：** 集計単位の違いが未充足率へどのように反映されるか．

**方法上の判断（Methodological decision）：** ルート，シナリオ，シードをそれぞれ等重みとする3種類の推定対象を分ける．

**分析行動（Action）：** 分子，分母，未充足率を集計単位別に算出する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`estimand_comparison.csv`と集計単位別の率．である．

**解釈上の範囲（Interpretation boundary）：** ルート加重では車両数が多い条件ほど寄与が大きい．異なる重み付けの率は同じ推定対象ではない．

**次の段階（Next step）：** 対応関係を保ったままシード間変動の区間を計算する．


# 20. ブートストラップ手続き

`CELL-ID: METHOD-BOOTSTRAP-01`

**目的：** 合成seedによる変動を定量化する
**入力：** seedでクラスタ化した全条件・全ルート
**処理：** 100 シード・クラスタを復元抽出し対応関係を保持する．1,000反復，percentile 2.5/97.5%，seed 20260711
**出力：** 制約集計と信頼区間
**検証：** 保存済み集計との一致
**解釈上の境界：** 区間はデータ取得・モデル・パラメータ・実運用の不確実性を含まない

# 20.1 ブートストラップ推定

`CELL-ID: METHOD-BOOTSTRAP-FORMAL-01`

同一シードから生成された顧客集合は，車両数・充電条件を横断して共有されるため，個々のroute-condition行を独立標本として再標本化しない．反復 $b$ ごとに100個のシード IDを復元抽出し，選択されたシードに属する全条件・全ルートを保持して $\hat p_c^{(b)}$ を再計算する．95%区間は1,000個の有限な反復値の2.5および97.5パーセンタイルである．

この区間が表すのは，凍結済み人口メッシュ，固定分布，固定パラメータ，固定ルートアルゴリズムの下でのシード・クラスタ変動である．データ取得誤差，需要分布のモデル不確実性，道路距離係数の妥当性，車両仕様の誤差，充電器の実利用可能性は区間に含まれない．したがって区間幅を研究全体の不確実性と解釈してはならない．

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-BOOTSTRAP-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 制約別未充足率を，ルート加重，シナリオ加重，シード加重に分けて算出した．

**残る疑問（Remaining question）：** 固定した生成モデルの下で，シード間変動をどの範囲として要約できるか．

**方法上の判断（Methodological decision）：** 同じシードに属する条件をまとめて再標本化するシード・クラスタ・ブートストラップを1,000回行う．

**分析行動（Action）：** 制約別のpercentile区間を算出する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`constraint_summary.csv`に保存される点推定値と区間下限・上限．である．

**解釈上の範囲（Interpretation boundary）：** 区間が表すのは100個の合成反復間の変動であり，モデル選択，パラメータ，実需要，交通，充電器可用性の不確実性は含まない．

**次の段階（Next step）：** 主要仮定を一つずつ変更して結果の感度を確認する．


In [ ]:
constraint_summary=cluster_bootstrap_constraint_summary(constraint_evaluations,case_rates,iterations=1000,random_seed=20260711)
constraint_summary.to_csv(SYNTH/'constraint_summary.csv',index=False)
stored_summary=pd.read_csv(required_files['stored_summary'][0]); summary_match,summary_match_detail=compare_frames(constraint_summary,stored_summary)
display(constraint_summary[['constraint_name','unmet_route_count','evaluated_route_count','route_weighted_unmet_rate','confidence_interval_lower','confidence_interval_upper']]); print(summary_match,summary_match_detail)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-BOOTSTRAP-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 制約別未充足率を，ルート加重，シナリオ加重，シード加重に分けて算出した．

**残る疑問（Remaining question）：** 固定した生成モデルの下で，シード間変動をどの範囲として要約できるか．

**方法上の判断（Methodological decision）：** 同じシードに属する条件をまとめて再標本化するシード・クラスタ・ブートストラップを1,000回行う．

**分析行動（Action）：** 制約別のpercentile区間を算出する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`constraint_summary.csv`に保存される点推定値と区間下限・上限．である．

**解釈上の範囲（Interpretation boundary）：** 区間が表すのは100個の合成反復間の変動であり，モデル選択，パラメータ，実需要，交通，充電器可用性の不確実性は含まない．

**次の段階（Next step）：** 主要仮定を一つずつ変更して結果の感度を確認する．


# 21. 感度分析

`CELL-ID: METHOD-SENSITIVITY-01`

**目的：** 一度に一つのパラメータを変えた応答を測る
**入力：** 再生成ルートとlow・base・highレジストリ
**処理：** 他のパラメータを固定して対象パラメータだけを変更する
**出力：** `sensitivity_detail.csv`
**検証：** 基準値との差と相対差を計算する
**解釈上の境界：** パラメータ間相互作用や較正不確実性は評価しない

# 21.1 感度推定量と分析範囲

`CELL-ID: METHOD-SENSITIVITY-FORMAL-01`

パラメータ $\theta_j$ の代替水準 $a$ に対する感度は，他のパラメータを基準値に固定した未充足率差 $\Delta_{c,j,a}=\hat p_c(\theta_j=a)-\hat p_c(\theta_j=base)$ として記録する．相対差は基準率が0でない場合のみ $\Delta_{c,j,a}/\hat p_c(base)$ とする．OATは局所的なモデル応答を可視化する設計であり，例えば速度と道路距離係数，航続距離と充電条件の相互作用を推定しない．また，代替水準の確率や現実性を評価しないため，感度の大きさは政策効果や因果効果ではない．

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-SENSITIVITY-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** シード・クラスタ・ブートストラップにより，固定モデル内の反復間変動を要約した．

**残る疑問（Remaining question）：** 基準値の未充足率が，選択した主要パラメータにどの程度依存するか．

**方法上の判断（Methodological decision）：** 他の条件を固定し，一度に一つのパラメータをlow/base/highへ変更するOAT分析を行う．

**分析行動（Action）：** 各設定の率と基準値との差を算出する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`sensitivity_detail.csv`と`sensitivity_summary.csv`．である．

**解釈上の範囲（Interpretation boundary）：** OAT分析は設定した範囲内の単独効果を示すが，パラメータ間の相互作用や範囲外の挙動は評価しない．

**次の段階（Next step）：** 運用要件と文献由来の回路幅参照値を区別して整理する．


In [ ]:
sensitivity_raw,parameter_response=run_oat_sensitivity(route_results,analysis_parameters,baseline_vehicle)
sensitivity_detail=sensitivity_raw.rename(columns={'parameter_value':'alternative_value','base_unmet_rate':'baseline_result','route_weighted_unmet_rate':'sensitivity_result','unmet_rate_change_from_base':'absolute_change'}).copy()
sensitivity_detail['baseline_value']=sensitivity_detail.parameter.map(analysis_parameters.set_index('parameter')['base'])
sensitivity_detail['unit']=sensitivity_detail.parameter.map(analysis_parameters.set_index('parameter')['unit'])
sensitivity_detail['direction_of_change']=sensitivity_detail.level
sensitivity_detail['affected_metric']=sensitivity_detail.constraint_name
sensitivity_detail['relative_change']=sensitivity_detail.absolute_change/sensitivity_detail.baseline_result.replace(0,np.nan)
sensitivity_detail['interpretation']='OAT model response under frozen synthetic data'
sensitivity_detail['limitation']='No parameter interaction or operational calibration'
sensitivity_detail.to_csv(TABLES/'sensitivity_detail.csv',index=False); display(sensitivity_detail.head(20))

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-SENSITIVITY-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** シード・クラスタ・ブートストラップにより，固定モデル内の反復間変動を要約した．

**残る疑問（Remaining question）：** 基準値の未充足率が，選択した主要パラメータにどの程度依存するか．

**方法上の判断（Methodological decision）：** 他の条件を固定し，一度に一つのパラメータをlow/base/highへ変更するOAT分析を行う．

**分析行動（Action）：** 各設定の率と基準値との差を算出する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`sensitivity_detail.csv`と`sensitivity_summary.csv`．である．

**解釈上の範囲（Interpretation boundary）：** OAT分析は設定した範囲内の単独効果を示すが，パラメータ間の相互作用や範囲外の挙動は評価しない．

**次の段階（Next step）：** 運用要件と文献由来の回路幅参照値を区別して整理する．


# 22. 量子回路幅の参照証拠

`CELL-ID: REFERENCE-CIRCUIT-01`

**目的：** 文献参照値と再現計算値を分離する
**入力：** スライド7・22およびローカル一次資料
**処理：** 定義と証拠状態を登録し，計算再現を主張しない
**出力：** `circuit_width_evidence.csv`
**検証：** 出典箇所未確認の値を`SOURCE_NOT_VERIFIED`とする
**解釈上の境界：** 回路幅・qubit数は計算費用全体や物理資源量と同一ではない

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-CIRCUIT-EVIDENCE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 選択した仮定の範囲について，制約別未充足率のOAT感度を算出した．

**残る疑問（Remaining question）：** 運用制約の分析結果と，量子VRP文献で報告された回路幅を混同せずに対応づけられるか．

**方法上の判断（Methodological decision）：** 回路幅を本シナリオの計算結果ではなく，出典確認状態を伴う参照証拠として登録する．

**分析行動（Action）：** 文献，値，定義，確認状態を証拠表へ記録する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`circuit_width_evidence.csv`と`SOURCE_NOT_VERIFIED`状態．である．

**解釈上の範囲（Interpretation boundary）：** 4件の回路幅はページ，式，インスタンス代入を確認できず，本Notebookの顧客・車両シナリオから導出した値ではない．

**次の段階（Next step）：** 再計算結果をスライド掲載値と照合する．


In [ ]:
circuit_evidence=pd.DataFrame([
('CW-L-MIN',128,'qubits','slide-described minimal formulation width','logical/not verified','unknown','time-window routing','not verified','minimal','not verified','Qubit-efficient quantum algorithms for VRP on NISQ processors','Leonidas et al.',2023,'https://arxiv.org/abs/2306.08507','MISSING','slide 7','slide extraction + local PDF search','Codex audit','SOURCE_NOT_VERIFIED'),
('CW-L-FULL',256,'qubits','slide-described full formulation width','logical/not verified','unknown','time-window routing','not verified','full','not verified','Qubit-efficient quantum algorithms for VRP on NISQ processors','Leonidas et al.',2023,'https://arxiv.org/abs/2306.08507','MISSING','slide 7','slide extraction + local PDF search','Codex audit','SOURCE_NOT_VERIFIED'),
('CW-O-HOBO',6080,'qubits','slide-described higher-order binary terms','logical/not verified','unknown','CVRP','not verified','higher-order','resource estimate','Requirements for Early Quantum Advantage and Utility in CVRP','Onah and Michielsen',2025,'https://arxiv.org/abs/2509.11469','MISSING','slide 7','slide extraction + local PDF search','Codex audit','SOURCE_NOT_VERIFIED'),
('CW-O-QUBO',14528,'qubits','slide-described quadratic binary terms','logical/not verified','unknown','CVRP','not verified','quadratic','resource estimate','Requirements for Early Quantum Advantage and Utility in CVRP','Onah and Michielsen',2025,'https://arxiv.org/abs/2509.11469','MISSING','slide 7','slide extraction + local PDF search','Codex audit','SOURCE_NOT_VERIFIED')],columns=['evidence_id','reported_value','unit','metric_definition','logical_or_physical','includes_ancilla','problem_type','instance_size','formulation','algorithm','source_title','authors','year','DOI_or_URL','page','table_or_figure','extraction_method','verified_by','status'])
circuit_evidence.to_csv(TABLES/'circuit_width_evidence.csv',index=False); display(circuit_evidence)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-CIRCUIT-EVIDENCE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 選択した仮定の範囲について，制約別未充足率のOAT感度を算出した．

**残る疑問（Remaining question）：** 運用制約の分析結果と，量子VRP文献で報告された回路幅を混同せずに対応づけられるか．

**方法上の判断（Methodological decision）：** 回路幅を本シナリオの計算結果ではなく，出典確認状態を伴う参照証拠として登録する．

**分析行動（Action）：** 文献，値，定義，確認状態を証拠表へ記録する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`circuit_width_evidence.csv`と`SOURCE_NOT_VERIFIED`状態．である．

**解釈上の範囲（Interpretation boundary）：** 4件の回路幅はページ，式，インスタンス代入を確認できず，本Notebookの顧客・車両シナリオから導出した値ではない．

**次の段階（Next step）：** 再計算結果をスライド掲載値と照合する．


# 23. 図表の生成

`CELL-ID: OUTPUT-REGISTRY-01`

**目的：** 図表を分類し解釈情報を保存する
**入力：** 実行済みデータオブジェクト
**処理：** 図表を生成し出力レジストリへ登録する
**出力：** 図および`figure_table_registry.csv`
**検証：** ファイルが存在し空でないこと
**解釈上の境界：** 概念図・参照値再描画を再現済み実証結果として扱わない

In [ ]:
fig,ax=plt.subplots(figsize=(9,4)); plot_data=constraint_summary.dropna(subset=['route_weighted_unmet_rate']); ax.barh(plot_data.constraint_name,plot_data.route_weighted_unmet_rate*100,facecolor='white',edgecolor='black',hatch='//'); ax.set(xlabel='Route-weighted unmet rate (%)',title='Reproduced model-conditional constraint results'); fig.tight_layout(); fig.savefig(FIGURES/'constraint_rates_reproduced.png',dpi=320); fig.savefig(FIGURES/'constraint_rates_reproduced.svg'); plt.show()
fig,ax=plt.subplots(figsize=(8,3.5)); ax.barh(circuit_evidence.evidence_id,circuit_evidence.reported_value,facecolor='none',edgecolor='black',hatch='xx'); ax.set_xscale('log'); ax.set(xlabel='Slide-reported qubit value (log scale; definition not verified)',title='Reference replotted — SOURCE_NOT_VERIFIED'); fig.tight_layout(); fig.savefig(FIGURES/'circuit_width_reference_replotted.png',dpi=320); fig.savefig(FIGURES/'circuit_width_reference_replotted.svg'); plt.show()
output_registry=pd.DataFrame([
('FIG-ROUTE','Representative route proxy','MODEL_DERIVED','route_edges; route_results','ROUTE-FIGURE-01','matplotlib','outputs/figures/representative_route_proxy.png','Shows algorithmic route proxy','Not road route/optimum'),
('FIG-CONSTRAINT','Constraint unmet rates','REPRODUCED','constraint_summary','FIGURE-GENERATE-01','matplotlib','outputs/figures/constraint_rates_reproduced.png','Shows route-weighted synthetic rates','Not operational failures'),
('FIG-CIRCUIT','Circuit width references','REFERENCE_REPLOTTED','circuit_evidence','FIGURE-GENERATE-01','matplotlib','outputs/figures/circuit_width_reference_replotted.png','Shows slide references','Source derivation unverified'),
('TABLE-PARAM','Parameter registry','DATA_DERIVED','source/config','PARAMETERS-CODE-01','pandas','outputs/tables/parameter_registry.csv','Central constants','Includes assumptions')],columns=['figure_id_or_table_id','title','classification','source_data','generating_cell','generating_function','output_path','interpretation','limitation'])
output_registry.to_csv(TABLES/'figure_table_registry.csv',index=False); display(output_registry)

# 24. スライド結果との照合

`CELL-ID: RECONCILIATION-01`

**目的：** Notebook内でスライド値との一致を計算する
**入力：** スライド参照値と再生成集計
**処理：** 結合，絶対差・相対差，丸め，0.05 percentage point許容差を計算する
**出力：** `result_reconciliation.csv`
**検証：** 状態を事前入力せず計算条件から判定する
**解釈上の境界：** 参照値との一致は外的妥当性を証明しない

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-RECONCILIATION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 文献由来の回路幅4値を，未確認の参照証拠として分離した．

**残る疑問（Remaining question）：** 再計算した未充足率が，スライドに掲載された丸め値と一致するか．

**方法上の判断（Methodological decision）：** 制約名で対応づけ，percentage point差と許容差を計算する．

**分析行動（Action）：** スライド値と再計算値を照合する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`result_reconciliation.csv`に保存される差分と照合状態．である．

**解釈上の範囲（Interpretation boundary）：** 数値の一致は同じ集計経路を再計算できたことを示すが，入力や仮定の科学的妥当性は示さない．

**次の段階（Next step）：** 件数，主キー，値，保存済みCSVとの一致を自動テストする．


In [ ]:
reference_values=pd.DataFrame([('M-PAYLOAD','CLM-S13',13,'Payload capacity',0.0),('M-TIME','CLM-S13',13,'Operating-time limit',33.5),('M-RANGE','CLM-S13',13,'Range feasibility',64.4),('M-SOC','CLM-S13',13,'SOC feasibility',np.nan),('M-ACCESS','CLM-S13',13,'Charging-station access',10.7),('M-ASSIST','CLM-S13',13,'Charging-assisted range feasibility',33.3),('M-DURATION','CLM-S13',13,'Charging-duration feasibility',15.3)],columns=['metric_id','claim_id','slide_number','constraint_name','reference_value'])
calculated=constraint_summary[['constraint_name','route_weighted_unmet_rate']].copy(); calculated['reproduced_value']=calculated.route_weighted_unmet_rate*100
reconciliation=reference_values.merge(calculated[['constraint_name','reproduced_value']],on='constraint_name',how='left'); reconciliation['unit']='percentage points'; reconciliation['rounding_rule']='slide displayed to 1 decimal'; reconciliation['tolerance']=0.05; reconciliation['absolute_difference']=(reconciliation.reproduced_value-reconciliation.reference_value).abs(); reconciliation['relative_difference']=reconciliation.absolute_difference/reconciliation.reference_value.abs().replace(0,np.nan); reconciliation['status']=np.where(reconciliation.constraint_name.eq('SOC feasibility'),'NOT_EVALUATED',np.where(reconciliation.absolute_difference.le(reconciliation.tolerance),'PASS','FAIL')); reconciliation['reason']=np.where(reconciliation.status.eq('PASS'),'recalculated value rounds to slide value',np.where(reconciliation.status.eq('NOT_EVALUATED'),'sequential SOC model absent','outside tolerance')); reconciliation['source_cell_id']='BOOTSTRAP-CODE-01'; reconciliation.to_csv(TABLES/'result_reconciliation.csv',index=False); display(reconciliation)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-RECONCILIATION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 文献由来の回路幅4値を，未確認の参照証拠として分離した．

**残る疑問（Remaining question）：** 再計算した未充足率が，スライドに掲載された丸め値と一致するか．

**方法上の判断（Methodological decision）：** 制約名で対応づけ，percentage point差と許容差を計算する．

**分析行動（Action）：** スライド値と再計算値を照合する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`result_reconciliation.csv`に保存される差分と照合状態．である．

**解釈上の範囲（Interpretation boundary）：** 数値の一致は同じ集計経路を再計算できたことを示すが，入力や仮定の科学的妥当性は示さない．

**次の段階（Next step）：** 件数，主キー，値，保存済みCSVとの一致を自動テストする．


# 25. 自動検証テスト

`CELL-ID: VALIDATION-01`

**目的：** 実際の条件式に基づく検証を実行する
**入力：** 全入力・生成オブジェクト
**処理：** 期待値と観測値を比較し状態とエラーを記録する
**出力：** `validation_summary.csv`
**検証：** ERRORレベルの失敗を最終判定へ反映する
**解釈上の境界：** 明示されたテスト範囲外の妥当性は保証しない

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-VALIDATION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 再計算値とスライドの丸め値について，差分と許容差を算出した．

**残る疑問（Remaining question）：** データ生成，ルート構築，集計，照合が，コードで定義した検査条件を満たすか．

**方法上の判断（Methodological decision）：** 期待件数，主キー一意性，値域，回帰的一致など18条件を式で評価する．

**分析行動（Action）：** 各条件の期待値，観測値，PASS/FAILを記録する．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`validation_summary.csv`に保存される18件の検査結果．である．

**解釈上の範囲（Interpretation boundary）：** 自動テストは実装の回帰的一貫性を検査するものであり，構成概念妥当性，最適性，外的妥当性は評価しない．

**次の段階（Next step）：** 検査結果と既知の未評価項目から再現性ラベルを決定する．


In [ ]:
tests=[]
add=lambda *args,**kwargs: tests.append(validation_row(*args,**kwargs))
add('T01','22 slides extracted',22,len(slides),len(slides)==22)
add('T02','input hashes match',0,int(data_provenance.hash_status.ne('MATCH').sum()),data_provenance.hash_status.eq('MATCH').all())
add('T03','scenario IDs unique',0,int(scenario_configurations.scenario_id.duplicated().sum()),not scenario_configurations.scenario_id.duplicated().any())
add('T04','scenario structures',27,len(scenario_configurations),len(scenario_configurations)==27)
add('T05','customer primary key unique',0,int(synthetic_customers.duplicated(['customer_configuration_id','customer_id']).sum()),not synthetic_customers.duplicated(['customer_configuration_id','customer_id']).any())
add('T06','customer rows',17500,len(synthetic_customers),len(synthetic_customers)==17500)
add('T07','all 100 seeds present',list(range(1,101)),sorted(synthetic_customers.seed.unique().tolist()),set(synthetic_customers.seed)==set(SEEDS))
add('T08','route-condition rows',8100,len(route_results),len(route_results)==8100)
route_count_check=base_routes.groupby(['customer_configuration_id','vehicle_count']).size().reset_index(name='routes'); add('T09','vehicle count equals routes per case',0,int((route_count_check.routes!=route_count_check.vehicle_count).sum()),route_count_check.routes.eq(route_count_check.vehicle_count).all())
add('T10','distances nonnegative',0,int(route_results.route_proxy_distance_km.lt(0).sum()),route_results.route_proxy_distance_km.ge(0).all())
add('T11','unmet numerator <= denominator',0,int((constraint_summary.unmet_route_count>constraint_summary.evaluated_route_count).sum()),(constraint_summary.unmet_route_count<=constraint_summary.evaluated_route_count).all())
add('T12','rates in [0,1]',0,int((~constraint_summary.route_weighted_unmet_rate.dropna().between(0,1)).sum()),constraint_summary.route_weighted_unmet_rate.dropna().between(0,1).all())
soc=constraint_summary.query('constraint_name=="SOC feasibility"').iloc[0]; add('T13','SOC not aggregated as 0%',True,bool(pd.isna(soc.route_weighted_unmet_rate) and soc.evaluated_route_count==0),pd.isna(soc.route_weighted_unmet_rate) and soc.evaluated_route_count==0)
add('T14','regenerated customers equal stored',True,customers_match,customers_match, error_message=customers_match_detail)
add('T15','regenerated routes equal stored',True,routes_match,routes_match,error_message=routes_match_detail)
add('T16','regenerated summary equal stored',True,summary_match,summary_match,error_message=summary_match_detail)
add('T17','slide reconciliation evaluated metrics pass',0,int(reconciliation.status.eq('FAIL').sum()),not reconciliation.status.eq('FAIL').any())
add('T18','required output figures generated',3,sum((FIGURES/name).is_file() for name in ['representative_route_proxy.png','constraint_rates_reproduced.png','circuit_width_reference_replotted.png']),all((FIGURES/name).is_file() and (FIGURES/name).stat().st_size>0 for name in ['representative_route_proxy.png','constraint_rates_reproduced.png','circuit_width_reference_replotted.png']))
validation_summary=pd.DataFrame(tests); validation_summary.to_csv(TABLES/'validation_summary.csv',index=False); display(validation_summary)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-VALIDATION-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 再計算値とスライドの丸め値について，差分と許容差を算出した．

**残る疑問（Remaining question）：** データ生成，ルート構築，集計，照合が，コードで定義した検査条件を満たすか．

**方法上の判断（Methodological decision）：** 期待件数，主キー一意性，値域，回帰的一致など18条件を式で評価する．

**分析行動（Action）：** 各条件の期待値，観測値，PASS/FAILを記録する．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`validation_summary.csv`に保存される18件の検査結果．である．

**解釈上の範囲（Interpretation boundary）：** 自動テストは実装の回帰的一貫性を検査するものであり，構成概念妥当性，最適性，外的妥当性は評価しない．

**次の段階（Next step）：** 検査結果と既知の未評価項目から再現性ラベルを決定する．


# 26. 結果

`CELL-ID: RESULTS-01`

再計算したルート加重未充足率を，分子，分母，シード・クラスタ・ブートストラップによるpercentile区間とともに示す．`SOC feasibility`は0%ではなく未評価（`NOT_EVALUATED`）である．以下の値は，固定した入力，合成規則，ルートプロキシ，パラメータの下で得られた推定値である．

再計算の結果，積載容量の未充足率は0.0%，運行時間は33.5185%，航続距離は64.4444%，充電アクセスは10.6914%，充電支援航続距離は33.2950%，充電時間は15.3021%であった．各値を小数第1位に丸めると，スライド掲載値との差は0.05 percentage point以内であった．SOCは評価対象となる分母が0であり，未充足率を算出していない．この照合結果は，凍結入力と現在の実装からスライド集計値を再計算できたことを示す．入力データ，閾値，ルートプロキシの科学的妥当性や，実運用への適合性は評価していない．

In [ ]:
display(constraint_summary[['constraint_name','unmet_route_count','evaluated_route_count','route_weighted_unmet_rate','confidence_interval_lower','confidence_interval_upper','main_assumption']])

# 27. 解釈

`CELL-ID: INTERPRETATION-01`

本シナリオの設定では，積載容量よりも運行時間と航続距離に関する未充足が多く観察された．ただし，積載容量が0%であったことは，一般の配送問題で容量制約が不要であることを意味しない．合成需要の上限と2,000 kgという容量の組合せでは，積載容量が未充足にならなかったという結果である．航続距離の64.4%は実際のEV配送の失敗率ではなく，Haversine距離を1.25倍したルートプロキシが，静的な81.2 kmの閾値を超えた割合である．

充電アクセスの10.7%は，分析用の充電候補までの地理的近接性に基づく．公共利用の可否，車両との互換性，稼働状況，混雑，営業時間は評価していない．充電支援条件による未充足率の変化についても，充電地点に到着した時点のSOCや逐次的なエネルギー収支を計算していないため，SOC実行可能性の証拠とはならない．時間，距離，充電を量子定式化へ追加する場合には，訪問順，資源状態，充電判断を表す変数や制約が増えると考えられるが，本Notebookはqubit数，回路深さ，ゲート数，補助変数，ペナルティ調整への増分を計算していない．文献に記載された回路幅が小さいことだけから，解品質，実行時間，ノイズ耐性，古典手法に対する優位性を判断することはできない．

# 27.1 研究者による振り返り（Researcher Reflection）

`CELL-ID: RESEARCHER-REFLECTION-01`

**当初の想定：** 当初の内的な想定を直接記録した研究メモは`NOT_DOCUMENTED`である．スライドから確認できるのは，問題規模と回路幅の比較だけではアプリケーション要件を解釈しにくいという問題設定である．

**分析で確認した事項：** 凍結入力と現在のルートプロキシの下では，積載容量の未充足は0件であり，運行時間，航続距離，簡略化した充電関連指標には未充足が生じた．SOCは未評価である．検査対象とした主キーおよび列について，再生成結果は保存済みCSVと一致した．`DIRECTLY_OBSERVABLE`

**想定との差：** 当初の想定と分析結果との差を判定できる同時期の記録は`NOT_DOCUMENTED`である．積載容量の未充足が0件であったことを研究者が予想外と捉えた証拠もない．

**認識している方法上の限界：** スライド13–16と現在のコードは，道路ネットワーク，観測需要，時間窓，逐次SOC，充電動態，古典最適化のbaseline，実運用による確認を分析範囲に含めていない．`CONTEMPORANEOUS_RECORD`

**次の分析で検討する変更：** スライド16は，運用上の現実性を高める方向と，application-stage framework（アプリケーションと技術段階を結ぶ枠組み）を構築する方向を提示している．優先順位は決定されていない．`CONTEMPORANEOUS_RECORD`

**維持する判断：** 未充足率を設定したモデルに依存する結果として扱い，SOCを未評価，回路幅を参照証拠，route proxyを非最適な近似経路として明記する．`DIRECTLY_OBSERVABLE`

**保留する判断：** 実配送への一般化，EV運用全体の実行可能性，量子資源の増分，quantum utility（量子的有用性），次段階の優先順位は，本Notebookから判断できないか未評価である．

# 28. 限界と妥当性への脅威

`CELL-ID: LIMITATIONS-01`

**構成概念妥当性：** 未充足率は，設定した制約とルートプロキシの関係を表す指標であり，配送の成功または失敗を直接測定していない．ルートプロキシは道路経路や最適化解ではなく，circuit width（回路幅）は量子計算に必要な資源全体を表さない．

**内的妥当性：** 結果は，シード，KMeansによる分割，最近傍のデポ選択，greedy訪問順，道路距離係数1.25，合成需要，合成サービス時間，一定速度，充電器属性の欠損処理に依存する．OAT分析はパラメータ間の相互作用を扱わず，ブートストラップ区間はモデル形式の不確実性を含まない．

**外的妥当性：** 東京都の公開データ，単一の基準車両，人口加重の顧客配置から得た結果を，他地域，季節，車種，配送事業者，観測交通条件へ直接適用することはできない．

**計算再現性：** 凍結した処理済み入力から同じ計算を再実行し，検査対象の結果と照合できる．一方，生データ取得時のリクエスト，車両仕様のURL，ライセンス情報の一部は不足している．動的APIの内容は取得時点によって変化し得る．回路幅4値は，該当ページ，式，インスタンスの代入過程を確認できないため，`SOURCE_NOT_VERIFIED`である．

# 28.1 研究過程の論理要約（Final Research Logic Summary）

`CELL-ID: FINAL-RESEARCH-LOGIC-01`

量子最適化の技術報告に示される問題規模や回路幅だけでは，輸送アプリケーションの運用要件がどこまで表現されているかを判断しにくいという観察から，量子資源を解釈する前に定義すべき運用要件を研究課題として設定した．その検討に用いる比較可能な輸送シナリオを構成するため，公開データから固定した入力，人口加重の合成顧客，デポとルートのプロキシ，個別の制約評価を採用した．具体的には，顧客生成，空間割当，距離・時間・充電に関する近似計算，シード・クラスタ・ブートストラップ，OAT感度分析を実行するコードを整備した．分析からは，制約別未充足率，区間推定，感度表，文献証拠表が得られる．これらは，本シナリオの設定内で運行時間や航続距離を分析対象から除外しない理由を示す一方，実配送の失敗率，全制約を同時に課したEVRPの実行可能性，最適性，quantum utility（量子的有用性）を示すものではない．次段階には，運用上の現実性を高める研究と，アプリケーション要件を量子技術段階へ接続する枠組みの検討が残る．問題設定と次段階の記述はスライドに残る同時期記録（`CONTEMPORANEOUS_RECORD`）に基づき，方法間の接続は事後的再構成（`RETROSPECTIVE_RECONSTRUCTION`）である．

# 29. 再現可能性の判定

`CELL-ID: FINAL-STATUS-01`

**目的：** 証拠に基づき最終的な再現可能性状態を導出する
**入力：** 事前検査，ハッシュ，動的検証，欠損・未評価項目
**処理：** 規則に基づき状態を分類する
**出力：** 最終状態
**検証：** ERROR失敗時は`EXECUTION_FAILED`とし，凍結入力からの再現を`FULLY_REPRODUCIBLE`としない
**解釈上の境界：** 判定対象は再現可能性であり研究結果の実質的妥当性ではない

### 分析行動の記録：実行前

`CELL-ID: TRAIL-BEFORE-FINAL-STATUS-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 入力ハッシュ，生成件数，主キー，保存済み結果，スライド値について，明示した検査条件を評価した．

**残る疑問（Remaining question）：** 得られた証拠の範囲に対応する再現性ラベルは何か．

**方法上の判断（Methodological decision）：** 重大な検査失敗，ハッシュ不一致，未確認資料，未評価項目を規則に従って分類する．

**分析行動（Action）：** 最終状態，理由，失敗件数，未評価項目を表にまとめる．

**作成・確認する証拠（Evidence produced）：** 実行時に作成する証拠は，`reproducibility_status.csv`と規則に基づく`final_status`．である．

**解釈上の範囲（Interpretation boundary）：** `COMPUTATIONALLY_REPRODUCIBLE_FROM_FROZEN_INPUTS`は凍結入力からの計算的一致を表し，生データ取得，経験的妥当性，運用妥当性まで含む取得過程を含む再現を意味しない．

**次の段階（Next step）：** 実行環境，時刻，Git状態，出力一覧をマニフェストへ記録する．


In [ ]:
critical_failures=validation_summary.query('severity=="ERROR" and status=="FAIL"')
hash_failures=int(data_provenance.hash_status.ne('MATCH').sum())
if len(critical_failures): final_status='EXECUTION_FAILED'
elif hash_failures: final_status='PARTIALLY_REPRODUCIBLE'
else: final_status='COMPUTATIONALLY_REPRODUCIBLE_FROM_FROZEN_INPUTS'
status_reason='All computational regeneration and comparison tests passed from exact frozen processed inputs; raw acquisition is incomplete, circuit references are unverified, and SOC is not evaluated.' if final_status.startswith('COMPUTATIONALLY') else 'See failed validation/provenance rows.'
final_status_table=pd.DataFrame([{'final_status':final_status,'reason':status_reason,'critical_failure_count':len(critical_failures),'hash_failure_count':hash_failures,'source_not_verified_count':int(circuit_evidence.status.eq('SOURCE_NOT_VERIFIED').sum()),'not_evaluated_items':'SOC feasibility'}]); display(final_status_table); final_status_table.to_csv(TABLES/'reproducibility_status.csv',index=False)

### 分析行動の記録：実行後

`CELL-ID: TRAIL-AFTER-FINAL-STATUS-CODE-01`
`STATUS: RETROSPECTIVE_RECONSTRUCTION`
`EVIDENCE CLASS: DIRECTLY_OBSERVABLE（コードと保存先）／RETROSPECTIVE_RECONSTRUCTION（判断間の接続）`

**直前までに確認した事項（Previous finding）：** 入力ハッシュ，生成件数，主キー，保存済み結果，スライド値について，明示した検査条件を評価した．

**残る疑問（Remaining question）：** 得られた証拠の範囲に対応する再現性ラベルは何か．

**方法上の判断（Methodological decision）：** 重大な検査失敗，ハッシュ不一致，未確認資料，未評価項目を規則に従って分類する．

**分析行動（Action）：** 最終状態，理由，失敗件数，未評価項目を表にまとめる．

**作成・確認する証拠（Evidence produced）：** 実行後の確認対象は，`reproducibility_status.csv`と規則に基づく`final_status`．である．

**解釈上の範囲（Interpretation boundary）：** `COMPUTATIONALLY_REPRODUCIBLE_FROM_FROZEN_INPUTS`は凍結入力からの計算的一致を表し，生データ取得，経験的妥当性，運用妥当性まで含む取得過程を含む再現を意味しない．

**次の段階（Next step）：** 実行環境，時刻，Git状態，出力一覧をマニフェストへ記録する．


# 30. 実行マニフェスト

`CELL-ID: RUN-MANIFEST-01`

**目的：** 実行識別情報と出力を記録する
**入力：** 実行状態，警告，生成ファイル
**処理：** 生成物のハッシュを計算しJSONへ保存する
**出力：** `outputs/manifests/run_manifest.json`
**検証：** 実行終了時にマニフェストと出力ハッシュを生成する
**解釈上の境界：** 実行済みNotebookとHTMLのハッシュはカーネル終了後に実行ラッパーが追加する

In [ ]:
END_TIME=datetime.now(timezone.utc)
generated_outputs=output_manifest(OUTPUTS)
generated_outputs.to_csv(MANIFESTS/'output_file_manifest.csv',index=False)
run_manifest={'scope':'Computational Reproduction from Frozen Processed Inputs and Audit Reconstruction','final_status':final_status,'execution_start_utc':START_TIME.isoformat(),'execution_end_utc':END_TIME.isoformat(),'duration_seconds':(END_TIME-START_TIME).total_seconds(),'python_version':sys.version,'operating_system':platform.platform(),'architecture':platform.machine(),'git_commit':git_info['git_commit'],'git_dirty':git_info['git_dirty'],'input_hash_status_counts':data_provenance.hash_status.value_counts().to_dict(),'validation_status_counts':validation_summary.status.value_counts().to_dict(),'warnings':'No warnings were globally suppressed; notebook-level captured warning count not implemented','errors':critical_failures.to_dict(orient='records'),'output_manifest':'outputs/manifests/output_file_manifest.csv'}
(MANIFESTS/'run_manifest.json').write_text(json.dumps(run_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([run_manifest]))